## Section 1 — Setup

In [ ]:
# 1.1 Environment check
import os, sys, platform
IN_COLAB = 'google.colab' in sys.modules
print('Running in Google Colab:', IN_COLAB, '| Python', platform.python_version())

import torch
print('PyTorch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Fine for Section 2; for training use Runtime > Change runtime type > T4 GPU.')

### 1.2 Connect Google Drive

In [ ]:
# 1.2 Mount Google Drive (Colab only)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not in Colab: skipping Drive mount (local paths are used, see 1.4).')

### 1.3 The %%shared helper — one copy of the code for notebook *and* app

In [ ]:
# 1.3 Register the %%shared cell magic
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

SHARED_CELLS = {}

@register_cell_magic
def shared(line, cell):
    name = line.strip() or f'cell_{len(SHARED_CELLS)}'
    SHARED_CELLS[name] = cell
    exec(compile(cell, f'<shared:{name}>', 'exec'), get_ipython().user_ns)

print('%%shared magic ready')

### 1.4 Imports and all settings

In [ ]:
%%shared header
# Imports and constants shared by the notebook and the Streamlit app  [shared with the app: header]
import json, math, time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
CACHE_SIZE = 320
MODEL_INPUT_SIZE = 224

In [ ]:
# 1.4b Notebook-only imports, settings and paths
import hashlib, random, shutil, subprocess, warnings, zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field, asdict
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

RUN_MODE = os.environ.get('TRUSTEYE_RUN_MODE', 'full')

@dataclass
class Config:
    seed: int = 42
    label_col: str = 'label_binary'
    expected_num_images: int = 3662
    cache_size: int = CACHE_SIZE
    cache_version: str = 'v1'
    png_compression: int = 3
    background_threshold: int = 10
    mask_work_size: int = 512
    min_foreground_fraction: float = 0.05
    dark_image_mean: float = 15.0
    dhash_size: int = 16
    dhash_candidate_max_distance: int = 30
    min_detail_correlation: float = 0.40
    detail_size: int = 128
    detail_blur_sigma: float = 2.0
    detail_edge_erode: int = 6
    split_fractions: dict = field(default_factory=lambda: {'train': 0.65, 'val1': 0.10, 'val2': 0.10, 'test': 0.15})
    io_threads: int = 4
    smoke_n_images: int = 240
    img_size: int = MODEL_INPUT_SIZE
    batch_size: int = 32
    epochs: int = 20
    patience: int = 5
    lr_head: float = 1e-3
    lr_backbone: float = 1e-4
    weight_decay: float = 1e-4
    unfreeze_from_stage: int = 7
    head_hidden: int = 512
    dropout: float = 0.2
    num_workers: int = 2
    max_batches_per_epoch: int = 0
    ablation_inputs: list = field(default_factory=lambda: ['rgb', 'afep_fused', 'afep_binary'])
    ablation_activations: list = field(default_factory=lambda: ['silu', 'mish'])
    ablation_seeds: list = field(default_factory=lambda: [42, 43, 44])
    ece_bins: int = 15
    ttda_n_max: int = 128
    ttda_n_grid: list = field(default_factory=lambda: [8, 16, 32, 64, 128])
    ttda_batch: int = 128
    tau_grid: list = field(default_factory=lambda: [0.80, 0.85, 0.90, 0.95, 0.97, 0.99])
    n_small_grid: list = field(default_factory=lambda: [16, 32])
    min_agreement: float = 0.99
    referral_rate: float = 0.10
    latency_images_gpu: int = 50
    latency_images_cpu: int = 10
    n_bootstrap: int = 1000
    idrid_kaggle_slug: str = ''

    def to_dict(self):
        return asdict(self)

CFG = Config()
if RUN_MODE == 'smoke':
    CFG.epochs, CFG.patience, CFG.max_batches_per_epoch, CFG.batch_size = 1, 1, 3, 16
    CFG.ablation_seeds = [42]
    CFG.ttda_n_max, CFG.ttda_n_grid, CFG.n_small_grid, CFG.ttda_batch = 8, [4, 8], [4], 32
    CFG.n_bootstrap, CFG.latency_images_gpu, CFG.latency_images_cpu = 50, 3, 2
if os.name == 'nt':
    CFG.num_workers = 0
CFG.expected_num_images = int(os.environ.get('TRUSTEYE_EXPECTED_N', CFG.expected_num_images))
CFG.idrid_kaggle_slug = os.environ.get('TRUSTEYE_IDRID_SLUG', CFG.idrid_kaggle_slug)

DRIVE_ROOT = Path(os.environ.get('TRUSTEYE_DRIVE_ROOT', '/content/drive/MyDrive/TrustEye-Edge'))
if RUN_MODE == 'smoke':
    DRIVE_ROOT = DRIVE_ROOT.with_name(DRIVE_ROOT.name + '-smoke')
LOCAL_DIR = Path(os.environ.get('TRUSTEYE_LOCAL_DIR', '/content/data'))
RAW_DIR = Path(os.environ.get('TRUSTEYE_RAW_DIR', '/content/aptos_raw'))
IDRID_RAW_DIR = Path(os.environ.get('TRUSTEYE_IDRID_RAW_DIR', '/content/idrid_raw'))

DATA_DIR = DRIVE_ROOT / 'data'
SPLIT_CSV = DRIVE_ROOT / 'splits' / f'aptos_split_{CFG.cache_version}.csv'
RUNS_DIR = DRIVE_ROOT / 'runs'
RESULTS_DIR = DRIVE_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
DEPLOY_DIR = DRIVE_ROOT / 'deploy'
CACHE_NAME = f'aptos_cache_{CFG.cache_size}_{CFG.cache_version}'
CACHE_ZIP = DATA_DIR / f'{CACHE_NAME}.zip'
MANIFEST_CSV = DATA_DIR / f'{CACHE_NAME}_manifest.csv'
VERIFY_JSON = DATA_DIR / 'data_verification_report.json'
DUP_DIR = DATA_DIR / 'duplicates'
IDRID_NAME = f'idrid_cache_{CFG.cache_size}_{CFG.cache_version}'
IDRID_ZIP = DATA_DIR / f'{IDRID_NAME}.zip'
IDRID_LABELS_CSV = DATA_DIR / 'idrid_labels.csv'
for d in (DATA_DIR / 'raw', DUP_DIR, SPLIT_CSV.parent, RUNS_DIR, RESULTS_DIR, FIG_DIR, DEPLOY_DIR, LOCAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'RUN_MODE={RUN_MODE} | device={DEVICE} | AMP={USE_AMP}')
print('Drive root :', DRIVE_ROOT)
print('Raw data   :', RAW_DIR)

### 1.5 Small helpers (seeding, fingerprints, JSON, run log)

In [ ]:
# 1.5 Small helpers (seeding, fingerprints, JSON, run log)
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def _json_default(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, (Path, torch.device)): return str(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))

def save_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=_json_default)
    os.replace(tmp, path)

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def environment_info():
    info = {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'numpy': np.__version__, 'opencv': cv2.__version__, 'device': str(DEVICE),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            'run_mode': RUN_MODE, 'timestamp': datetime.now().isoformat(timespec='seconds')}
    return info

def run_cmd(cmd):
    print('$', ' '.join(map(str, cmd)))
    res = subprocess.run(list(map(str, cmd)), capture_output=True, text=True)
    print(res.stdout[-2000:], res.stderr[-2000:])
    if res.returncode != 0:
        raise RuntimeError(f'Command failed ({res.returncode}): {cmd}')

def savefig(fig, name):
    fig.tight_layout(); fig.savefig(FIG_DIR / name, dpi=120, bbox_inches='tight'); plt.show()

set_seed(CFG.seed)
print('helpers ready')

## Section 2 — P0: dataset verification and preparation (no training)

In [ ]:
# 2.0 Has P0 already been completed (in an earlier session)?
P0_DONE = CACHE_ZIP.exists() and MANIFEST_CSV.exists() and SPLIT_CSV.exists()
print('P0 already completed on Drive:', P0_DONE)
if P0_DONE:
    print('-> Sections 2.2-2.8 will skip their work; go on running cells, nothing is rebuilt.')

### 2.1 Kaggle credentials via Colab Secrets

In [ ]:
# 2.1 Kaggle credentials via Colab Secrets
def kaggle_auth():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return
    if not IN_COLAB:
        raise RuntimeError('Kaggle credentials missing: set KAGGLE_USERNAME and KAGGLE_KEY environment variables.')
    from google.colab import userdata
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except Exception as e:
        raise RuntimeError('Add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (key icon) with notebook access.') from e
    print('Kaggle credentials loaded from Colab Secrets.')

### 2.2 Make sure the raw APTOS data is in /content/aptos_raw

In [ ]:
# 2.2 Make sure the raw APTOS data is in /content/aptos_raw
def count_raw_images():
    folder = RAW_DIR / 'train_images'
    return len(list(folder.glob('*.png'))) if folder.exists() else 0

def ensure_aptos_raw():
    if (RAW_DIR / 'train.csv').exists() and count_raw_images() == CFG.expected_num_images:
        print(f'Raw data present: {count_raw_images()} images + train.csv')
        return
    kaggle_auth()
    archive = Path('/content/aptos2019-blindness-detection.zip')
    if not archive.exists():
        run_cmd(['kaggle', 'competitions', 'download', '-c', 'aptos2019-blindness-detection', '-p', '/content'])
    with zipfile.ZipFile(archive) as zf:
        members = [m for m in zf.namelist() if m.startswith('train_images/') or m == 'train.csv']
        for m in tqdm(members, desc='Extracting train images'):
            zf.extract(m, RAW_DIR)
    archive.unlink()
    print(f'Raw data ready: {count_raw_images()} images')

if not P0_DONE:
    ensure_aptos_raw()
else:
    print('P0 done - raw data not needed.')

### 2.3 Fundus cropping (shared with the app)

In [ ]:
%%shared fundus
# 2.3 Fundus cropping (shared with the app)  [shared with the app: fundus]
def to_bgr_uint8(image):
    notes = []
    if image.dtype == np.uint16:
        image = (image / 257.0).round().astype(np.uint8); notes.append('converted_16bit')
    elif image.dtype != np.uint8:
        raise ValueError(f'Unsupported pixel type {image.dtype}')
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR); notes.append('converted_gray')
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR); notes.append('dropped_alpha')
    return image, notes

def find_fundus_box(bgr, background_threshold=10, work_size=512):
    h, w = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    scale = min(1.0, work_size / max(h, w))
    small = gray if scale == 1.0 else cv2.resize(gray, (max(1, round(w * scale)), max(1, round(h * scale))),
                                                 interpolation=cv2.INTER_AREA)
    mask = cv2.morphologyEx((small > background_threshold).astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, _, st, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return (0, 0, w, h), 0.0
    k = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    x, y, bw, bh, area = st[k]
    fg = float(area) / float(small.shape[0] * small.shape[1])
    box = (max(0, math.floor(x / scale)), max(0, math.floor(y / scale)),
           min(w, math.ceil((x + bw) / scale)), min(h, math.ceil((y + bh) / scale)))
    return box, fg

def pad_to_square(bgr):
    h, w = bgr.shape[:2]; s = max(h, w); top, left = (s - h) // 2, (s - w) // 2
    return cv2.copyMakeBorder(bgr, top, s - h - top, left, s - w - left, cv2.BORDER_CONSTANT, value=(0, 0, 0))

def preprocess_fundus(bgr, size=CACHE_SIZE, background_threshold=10, work_size=512, min_foreground_fraction=0.05):
    (x0, y0, x1, y1), fg = find_fundus_box(bgr, background_threshold, work_size)
    crop_ok = fg >= min_foreground_fraction
    if not crop_ok:
        x0, y0, x1, y1 = 0, 0, bgr.shape[1], bgr.shape[0]
    out = cv2.resize(pad_to_square(bgr[y0:y1, x0:x1]), (size, size), interpolation=cv2.INTER_AREA)
    return out, {'crop_x0': x0, 'crop_y0': y0, 'crop_x1': x1, 'crop_y1': y1,
                 'foreground_fraction': round(fg, 4), 'crop_ok': bool(crop_ok)}

### 2.4 Duplicate-detection tools (two stages)

In [ ]:
# 2.4 Duplicate-detection tools (two stages)
_POPCOUNT = np.array([bin(b).count('1') for b in range(256)], dtype=np.uint8)

def dhash(bgr, hash_size=16):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (hash_size + 1, hash_size), interpolation=cv2.INTER_AREA)
    return np.packbits((small[:, 1:] > small[:, :-1]).flatten()).tobytes().hex()

def hamming_matrix(hex_hashes):
    arr = np.stack([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in hex_hashes])
    dist = np.zeros((len(arr), len(arr)), dtype=np.int16)
    for i in range(len(arr)):
        dist[i] = _POPCOUNT[np.bitwise_xor(arr, arr[i])].sum(axis=1)
    return dist

def detail_signature(bgr, size=128, blur_sigma=2.0, background_threshold=10, edge_erode=6):
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32)
    detail = small - cv2.GaussianBlur(small, (0, 0), blur_sigma)
    inside = cv2.erode((small > background_threshold).astype(np.uint8), np.ones((3, 3), np.uint8),
                       iterations=edge_erode).astype(bool)
    detail[~inside] = 0.0
    detail[inside] -= detail[inside].mean()
    n = np.linalg.norm(detail)
    return (detail / n).ravel() if n > 0 else detail.ravel()

def candidate_pairs(dist, ids, max_distance):
    r, c = np.nonzero(np.triu(dist <= max_distance, k=1))
    return pd.DataFrame({'id_a': [ids[i] for i in r], 'id_b': [ids[j] for j in c],
                         'distance': dist[r, c]}).sort_values(['distance', 'id_a', 'id_b'], ignore_index=True)

def add_detail_correlation(pairs, image_dir):
    sig = {}
    def get(i):
        if i not in sig:
            sig[i] = detail_signature(cv2.imread(str(Path(image_dir) / f'{i}.png')), CFG.detail_size,
                                      CFG.detail_blur_sigma, CFG.background_threshold, CFG.detail_edge_erode)
        return sig[i]
    pairs = pairs.copy()
    pairs['detail_corr'] = [float(get(a) @ get(b)) for a, b in tqdm(zip(pairs['id_a'], pairs['id_b']),
                                                                    total=len(pairs), desc='Stage-2 correlation')]
    return pairs

def exact_duplicate_pairs(manifest):
    out = []
    for _, g in manifest.groupby('pixel_sha256'):
        ids = sorted(g['id_code'])
        out += [(ids[i], ids[j]) for i in range(len(ids)) for j in range(i + 1, len(ids))]
    return pd.DataFrame(out, columns=['id_a', 'id_b'])

def build_groups(ids, pairs):
    parent = {i: i for i in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for a, b in zip(pairs['id_a'], pairs['id_b']):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    roots = {i: find(i) for i in ids}
    names = {r: f'g{k:04d}' for k, r in enumerate(sorted(set(roots.values())))}
    return pd.Series({i: names[roots[i]] for i in ids}, name='dup_group')

def summarize_groups(df):
    rows = []
    for g, grp in df.groupby('dup_group'):
        if len(grp) < 2:
            continue
        grades = sorted(grp['diagnosis'].astype(int))
        rows.append({'dup_group': g, 'size': len(grp), 'ids': ';'.join(sorted(grp['id_code'])),
                     'grades': ';'.join(map(str, grades)), 'grade_conflict': len(set(grades)) > 1,
                     'binary_conflict': len({min(x, 1) for x in grades}) > 1})
    cols = ['dup_group', 'size', 'ids', 'grades', 'grade_conflict', 'binary_conflict']
    return pd.DataFrame(rows, columns=cols).sort_values('size', ascending=False, ignore_index=True)
print('duplicate tools ready')

### 2.5 Verification + cache-building tools

In [ ]:
# 2.5 Verification + cache-building tools
def verify_labels(csv_path, image_dir, expected):
    df = pd.read_csv(csv_path, dtype={'id_code': str})
    errors = []
    if not {'id_code', 'diagnosis'} <= set(df.columns):
        return df, {'ok': False, 'errors': [f'missing columns, found {list(df.columns)}']}
    if df[['id_code', 'diagnosis']].isna().any().any(): errors.append('empty cells')
    if df['id_code'].duplicated().any(): errors.append(f"{df['id_code'].duplicated().sum()} repeated ids")
    bad = sorted(set(df['diagnosis'].dropna().astype(int)) - {0, 1, 2, 3, 4})
    if bad: errors.append(f'grades outside 0-4: {bad}')
    img_ids, csv_ids = {p.stem for p in Path(image_dir).glob('*.png')}, set(df['id_code'])
    if csv_ids - img_ids: errors.append(f'{len(csv_ids - img_ids)} ids without image')
    if img_ids - csv_ids: errors.append(f'{len(img_ids - csv_ids)} images without label')
    if len(df) != expected: errors.append(f'expected {expected} rows, found {len(df)}')
    df['diagnosis'] = df['diagnosis'].astype(int)
    return df, {'ok': not errors, 'errors': errors, 'n_rows': len(df),
                'grade_counts': {int(k): int(v) for k, v in df['diagnosis'].value_counts().sort_index().items()},
                'binary_counts': {'no_dr_0': int((df['diagnosis'] == 0).sum()), 'dr_1to4': int((df['diagnosis'] >= 1).sum())}}

def process_one_image(id_code, src_path, dst_path):
    row = {'id_code': id_code}
    try:
        raw = Path(src_path).read_bytes()
        row.update(file_bytes=len(raw), file_sha256=sha256_bytes(raw))
        img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError('OpenCV could not decode the file')
        row.update(orig_height=img.shape[0], orig_width=img.shape[1],
                   orig_channels=1 if img.ndim == 2 else img.shape[2], orig_dtype=str(img.dtype),
                   pixel_sha256=sha256_bytes(str(img.shape).encode() + img.tobytes()))
        bgr, notes = to_bgr_uint8(img)
        mean_gray = float(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean())
        out, info = preprocess_fundus(bgr, CFG.cache_size, CFG.background_threshold, CFG.mask_work_size,
                                      CFG.min_foreground_fraction)
        ok, enc = cv2.imencode('.png', out, [cv2.IMWRITE_PNG_COMPRESSION, CFG.png_compression])
        if not ok:
            raise ValueError('PNG encoding failed')
        Path(dst_path).write_bytes(enc.tobytes())
        flags = list(notes)
        if not info['crop_ok']: flags.append('crop_unreliable')
        if mean_gray < CFG.dark_image_mean: flags.append('very_dark')
        if min(img.shape[:2]) < CFG.cache_size: flags.append('smaller_than_cache')
        row.update(info, mean_intensity=round(mean_gray, 2), cache_sha256=sha256_bytes(enc.tobytes()),
                   dhash=dhash(out, CFG.dhash_size), flags=';'.join(flags), decode_ok=True, error='')
    except Exception as e:
        row.update(decode_ok=False, error=f'{type(e).__name__}: {e}')
    return row

def build_cache(items, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(r.id_code, r.src_path, out_dir / f'{r.id_code}.png') for r in items.itertuples()]
    with ThreadPoolExecutor(CFG.io_threads) as pool:
        rows = list(tqdm(pool.map(lambda j: process_one_image(*j), jobs), total=len(jobs), desc='Verify + cache'))
    return pd.DataFrame(rows)

def describe(s):
    s = s.astype(float); return {'min': round(s.min(), 2), 'median': round(s.median(), 2), 'max': round(s.max(), 2)}

def summarize_manifest(m):
    ok = m[m['decode_ok']]
    flags = ok['flags'].fillna('').str.split(';').explode(); flags = flags[flags != '']
    return {'n_total': len(m), 'n_decoded_ok': len(ok), 'n_decode_failed': int((~m['decode_ok']).sum()),
            'failed_ids': m.loc[~m['decode_ok'], 'id_code'].tolist(),
            'flag_counts': {k: int(v) for k, v in flags.value_counts().items()},
            'orig_width': describe(ok['orig_width']), 'orig_height': describe(ok['orig_height']),
            'file_megabytes': describe(ok['file_bytes'] / 1e6),
            'n_unique_pixel_sha256': int(ok['pixel_sha256'].nunique())}

def zip_folder(folder, zip_path):
    files = sorted(Path(folder).glob('*.png'))
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as zf:
        for f in tqdm(files, desc=f'Zipping {Path(folder).name}'):
            zf.write(f, arcname=f'{Path(folder).name}/{f.name}')
    return len(files)

def copy_zip_to_drive(local_zip, drive_zip, n_files):
    h = sha256_file(local_zip)
    shutil.copy2(local_zip, drive_zip)
    if sha256_file(drive_zip) != h:
        raise IOError('Drive copy differs from local zip - re-run this cell.')
    rec = {'file': Path(drive_zip).name, 'sha256': h, 'megabytes': round(Path(drive_zip).stat().st_size / 1e6, 1),
           'n_files': int(n_files)}
    save_json(rec, str(drive_zip) + '.sha256.json')
    return rec

def restore_cache_from_drive(drive_zip):
    rec = load_json(str(drive_zip) + '.sha256.json')
    target = LOCAL_DIR / Path(drive_zip).stem
    if target.exists() and len(list(target.glob('*.png'))) == rec['n_files']:
        print(f'Cache already on local disk: {target} ({rec["n_files"]} files)'); return target
    local_zip = LOCAL_DIR / Path(drive_zip).name
    print(f'Copying {rec["megabytes"]} MB from Drive ...'); shutil.copy2(drive_zip, local_zip)
    if sha256_file(local_zip) != rec['sha256']:
        raise IOError('Checksum mismatch: the zip copied from Drive is corrupted.')
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(LOCAL_DIR)
    local_zip.unlink()
    n = len(list(target.glob('*.png')))
    if n != rec['n_files']:
        raise IOError(f'Expected {rec["n_files"]} images, found {n}')
    print(f'Cache restored: {target} ({n} files, checksum OK)')
    return target

def read_rgb(path, max_side=None):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    if max_side and max(bgr.shape[:2]) > max_side:
        s = max_side / max(bgr.shape[:2])
        bgr = cv2.resize(bgr, (round(bgr.shape[1] * s), round(bgr.shape[0] * s)), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def show_grid(images, titles, ncols=4, cell=3.0, title='', save=None):
    nrows = max(1, int(np.ceil(len(images) / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(cell * ncols, cell * nrows), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, im, t in zip(axes.flat, images, titles):
        ax.imshow(im, cmap='gray' if im.ndim == 2 else None); ax.set_title(t, fontsize=7)
    if title: fig.suptitle(title)
    if save: savefig(fig, save)
    else: plt.show()
print('verification tools ready')

### 2.6 Run P0.1–P0.3: verify labels, verify every image, build and save the cache

In [ ]:
# 2.6 Run P0.1–P0.3: verify labels, verify every image, build and save the cache
LOCAL_CACHE = LOCAL_DIR / CACHE_NAME
if P0_DONE:
    print('P0 done - skipping verification and cache building.')
else:
    labels, label_report = verify_labels(RAW_DIR / 'train.csv', RAW_DIR / 'train_images', CFG.expected_num_images)
    print(json.dumps(label_report, indent=2))
    assert label_report['ok'], 'Label verification FAILED - share this output.'
    shutil.copy2(RAW_DIR / 'train.csv', DATA_DIR / 'raw' / 'train.csv')

    if RUN_MODE == 'smoke':
        labels, _ = train_test_split(labels, train_size=CFG.smoke_n_images, stratify=labels['diagnosis'],
                                     random_state=CFG.seed)
        print('SMOKE MODE: using', len(labels), 'images')

    items = pd.DataFrame({'id_code': labels['id_code'].values})
    items['src_path'] = [str(RAW_DIR / 'train_images' / f'{i}.png') for i in items['id_code']]
    t0 = time.time()
    manifest = build_cache(items, LOCAL_CACHE).merge(labels, on='id_code', how='left')
    build_minutes = round((time.time() - t0) / 60, 2)
    image_report = summarize_manifest(manifest)
    print(f'Cache built in {build_minutes} min'); print(json.dumps(image_report, indent=2))

    ok_ids = manifest.loc[manifest['decode_ok'], 'id_code']
    pick = list(ok_ids.sample(3, random_state=CFG.seed)) + \
           list(manifest[manifest['decode_ok']].nsmallest(3, 'foreground_fraction')['id_code'])
    ims, tts = [], []
    for i in pick:
        ims += [read_rgb(RAW_DIR / 'train_images' / f'{i}.png', 400), read_rgb(LOCAL_CACHE / f'{i}.png')]
        tts += [f'{i} original (shrunk)', f'{i} cached 320x320']
    show_grid(ims, tts, ncols=4, title='Crop check: random (first 3) and hardest (last 3)', save='p0_crop_check.png')

    n_files = zip_folder(LOCAL_CACHE, LOCAL_DIR / f'{CACHE_NAME}.zip')
    cache_record = copy_zip_to_drive(LOCAL_DIR / f'{CACHE_NAME}.zip', CACHE_ZIP, n_files)
    manifest.to_csv(MANIFEST_CSV, index=False)
    save_json({'labels': label_report, 'images': image_report, 'cache': cache_record, 'build_minutes': build_minutes,
               'config': CFG.to_dict(), 'environment': environment_info()}, VERIFY_JSON)
    print('Saved to Drive:', CACHE_ZIP.name, '|', MANIFEST_CSV.name, '|', VERIFY_JSON.name)

### 2.7 Run P0.4: duplicate detection and groups

In [ ]:
# 2.7 Run P0.4: duplicate detection and groups
def show_pairs(pairs, image_dir, grades, n, title, save):
    pairs = pairs.head(n)
    if pairs.empty:
        print(f'[{title}] no pairs'); return
    ims, tts = [], []
    for r in pairs.itertuples():
        for i in (r.id_a, r.id_b):
            ims.append(read_rgb(Path(image_dir) / f'{i}.png')); tts.append(f'{i} g={grades.get(i, "?")} d={r.distance} c={r.detail_corr:.2f}')
    show_grid(ims, tts, ncols=4, cell=2.6, title=title, save=save)

if P0_DONE:
    print('P0 done - skipping duplicate detection (results are in data/duplicates/ on Drive).')
else:
    ok = manifest[manifest['decode_ok']].reset_index(drop=True)
    ids = ok['id_code'].tolist()
    dist = hamming_matrix(ok['dhash'].tolist())
    cand = add_detail_correlation(candidate_pairs(dist, ids, CFG.dhash_candidate_max_distance), LOCAL_CACHE)
    exact = exact_duplicate_pairs(ok)
    print(f'stage-1 candidates: {len(cand)} | exact pixel-identical pairs: {len(exact)}')
    print(pd.crosstab(pd.cut(cand['distance'], [-1, 0, 4, 8, 16, 24, 30]),
                      pd.cut(cand['detail_corr'], [-1, 0.2, 0.3, 0.4, 0.6, 0.8, 0.9, 1.01])).to_string())

    grades = ok.set_index('id_code')['diagnosis']; thr = CFG.min_detail_correlation
    show_pairs(cand[cand['detail_corr'] >= 0.9], LOCAL_CACHE, grades, 2, 'A: corr >= 0.9', 'p0_pairs_A.png')
    show_pairs(cand[(cand['detail_corr'] >= thr) & (cand['detail_corr'] < 0.9)].sort_values('detail_corr'),
               LOCAL_CACHE, grades, 6, 'B: threshold..0.9 (grouped)', 'p0_pairs_B.png')
    show_pairs(cand[cand['detail_corr'] < thr].sort_values('detail_corr', ascending=False),
               LOCAL_CACHE, grades, 4, 'C: just below threshold (NOT grouped)', 'p0_pairs_C.png')

    near = cand.loc[cand['detail_corr'] >= thr, ['id_a', 'id_b']].assign(reason='near')
    confirmed = pd.concat([exact.assign(reason='exact'), near]).drop_duplicates(['id_a', 'id_b']).reset_index(drop=True)
    ok['dup_group'] = ok['id_code'].map(build_groups(ids, confirmed))
    groups = summarize_groups(ok)
    cand.to_csv(DUP_DIR / 'candidate_pairs.csv', index=False)
    confirmed.to_csv(DUP_DIR / 'confirmed_pairs.csv', index=False)
    groups.to_csv(DUP_DIR / 'duplicate_groups.csv', index=False)
    dup_stats = {'n_exact_pairs': len(exact), 'n_confirmed_pairs': len(confirmed),
                 'by_reason': confirmed['reason'].value_counts().to_dict(), 'n_groups': len(groups),
                 'n_images_in_groups': int(groups['size'].sum()), 'n_grade_conflicts': int(groups['grade_conflict'].sum()),
                 'n_binary_conflicts': int(groups['binary_conflict'].sum())}
    save_json(dup_stats, DUP_DIR / 'duplicate_stats.json')
    print(json.dumps(dup_stats, indent=2))
    print(groups[groups['binary_conflict']].to_string(index=False))

### 2.8 Run P0.5: group-aware, stratified split (65 / 10 / 10 / 15) — then frozen

In [ ]:
# 2.8 Run P0.5: group-aware, stratified split (65 / 10 / 10 / 15) — then frozen
def make_split(df, fractions, seed):
    counts = {k: round(v * 20) for k, v in fractions.items()}
    assert sum(counts.values()) == 20, counts
    fold = np.full(len(df), -1)
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    for f, (_, idx) in enumerate(sgkf.split(df, df['diagnosis'], groups=df['dup_group'])):
        fold[idx] = f
    order = [s for s in ['test', 'val2', 'val1', 'train'] for _ in range(counts[s])]
    return pd.Series([order[f] for f in fold], index=df.index)

def check_split(df):
    leaking = df.groupby('dup_group')['split'].nunique().loc[lambda s: s > 1].index.tolist()
    return {'ok': not leaking and df['id_code'].is_unique, 'leaking_groups': leaking,
            'counts': df['split'].value_counts().to_dict(),
            'fractions': df['split'].value_counts(normalize=True).round(4).to_dict(),
            'dr_fraction': df.groupby('split')['label_binary'].mean().round(4).to_dict()}

if P0_DONE:
    split_df = pd.read_csv(SPLIT_CSV, dtype={'id_code': str})
    print('Loaded frozen split:', split_df['split'].value_counts().to_dict())
else:
    split_df = ok[['id_code', 'diagnosis', 'dup_group']].copy()
    split_df['label_binary'] = (split_df['diagnosis'] >= 1).astype(int)
    split_df['label_referable'] = (split_df['diagnosis'] >= 2).astype(int)
    split_df['split'] = make_split(split_df, CFG.split_fractions, CFG.seed)
    rep = check_split(split_df)
    print(json.dumps(rep, indent=2))
    assert rep['ok'], 'LEAKAGE CHECK FAILED'
    split_df.sort_values('id_code').to_csv(SPLIT_CSV, index=False)
    save_json(rep, RESULTS_DIR / 'p0_split_report.json')
    P0_DONE = True
    print('Split saved and frozen:', SPLIT_CSV)
print(pd.crosstab(split_df['split'], split_df['diagnosis'], margins=True))

### 2.9 IDRiD external test set

In [ ]:
# 2.9 IDRiD external test set
def _subset(path):
    t = str(path).lower(); tr, te = 'train' in t, 'test' in t
    return 'train' if tr and not te else 'test' if te and not tr else 'unknown'

def find_idrid_labels(root):
    tables = []
    for csv in sorted(Path(root).rglob('*.csv')):
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue
        cols = {' '.join(str(c).strip().lower().replace('_', ' ').split()): c for c in df.columns}
        name = next((cols[c] for c in cols if 'image' in c and ('name' in c or 'id' in c)), None)
        grade = next((cols[c] for c in cols if 'retinopathy' in c), None)
        if name is None or grade is None:
            continue
        t = pd.DataFrame({'image_name': df[name].astype(str).str.strip(), 'grade': pd.to_numeric(df[grade], errors='coerce')})
        t = t[(t['image_name'] != '') & (t['image_name'] != 'nan')].dropna(subset=['grade'])
        t['grade'] = t['grade'].astype(int); t['subset'] = _subset(csv.relative_to(root)); tables.append(t)
    if not tables:
        raise FileNotFoundError('No IDRiD grading CSV found - share the folder listing printed above.')
    return pd.concat(tables, ignore_index=True)

def match_idrid(labels, root):
    imgs = pd.DataFrame([{'path': str(p), 'stem': p.stem.lower(), 'subset': _subset(p.relative_to(root))}
                         for p in Path(root).rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}])
    keyed = imgs.groupby(['subset', 'stem'])['path'].apply(list)
    labels = labels.copy()
    labels['stem'] = labels['image_name'].str.lower().str.replace(r'\.(jpe?g|png|tiff?)$', '', regex=True)
    def pick(c):
        if len(c) == 1: return c[0]
        g = [x for x in c if 'grading' in x.lower()]
        return g[0] if len(g) == 1 else None
    labels['src_path'] = [pick(keyed.get((r.subset, r.stem), [])) for r in labels.itertuples()]
    m = labels.dropna(subset=['src_path']).copy()
    m['id_code'] = 'idrid_' + m['subset'] + '_' + m['stem']
    return m.reset_index(drop=True), int(labels['src_path'].isna().sum())

if IDRID_ZIP.exists():
    print('IDRiD cache already on Drive - nothing to do.')
elif not CFG.idrid_kaggle_slug and not IDRID_RAW_DIR.exists():
    print('IDRiD skipped (no Kaggle dataset id set). External testing in Section 11 will be skipped.')
else:
    if not IDRID_RAW_DIR.exists():
        kaggle_auth()
        run_cmd(['kaggle', 'datasets', 'download', '-d', CFG.idrid_kaggle_slug, '-p', IDRID_RAW_DIR, '--unzip'])
    for d in sorted({p.parent for p in IDRID_RAW_DIR.rglob('*') if p.is_file()})[:40]:
        print(f'{d.relative_to(IDRID_RAW_DIR)}  ({sum(1 for _ in d.iterdir())} items)')
    idrid_matched, n_unmatched = match_idrid(find_idrid_labels(IDRID_RAW_DIR), IDRID_RAW_DIR)
    print(f'matched {len(idrid_matched)} images, unmatched/ambiguous label rows: {n_unmatched}')
    print(idrid_matched.groupby(['subset', 'grade']).size().unstack(fill_value=0))
    assert idrid_matched['id_code'].is_unique and len(idrid_matched) > 0
    if len(idrid_matched) != 516:
        print('NOTE: the official disease-grading set has 516 images - check this copy before using it.')
    im = build_cache(idrid_matched[['id_code', 'src_path']], LOCAL_DIR / IDRID_NAME)
    im = im.merge(idrid_matched[['id_code', 'image_name', 'subset', 'grade']], on='id_code')
    print(json.dumps(summarize_manifest(im), indent=2))
    okm = im[im['decode_ok']].copy()
    okm['label_binary'] = (okm['grade'] >= 1).astype(int)
    okm[['id_code', 'image_name', 'subset', 'grade', 'label_binary']].to_csv(IDRID_LABELS_CSV, index=False)
    im.to_csv(DATA_DIR / f'{IDRID_NAME}_manifest.csv', index=False)
    n = zip_folder(LOCAL_DIR / IDRID_NAME, LOCAL_DIR / f'{IDRID_NAME}.zip')
    print(copy_zip_to_drive(LOCAL_DIR / f'{IDRID_NAME}.zip', IDRID_ZIP, n))

### 2.10 Every session: restore the cache and load images into memory

In [ ]:
# 2.10 Every session: restore the cache and load images into memory
LOCAL_CACHE = restore_cache_from_drive(CACHE_ZIP)
split_df = pd.read_csv(SPLIT_CSV, dtype={'id_code': str})
assert len(set(split_df['id_code']) - {p.stem for p in LOCAL_CACHE.glob('*.png')}) == 0, 'cache/split mismatch'

_IMAGE_CACHE = {}
def load_split(split):
    if split == 'test' and not globals().get('FINAL_EVALUATION_UNLOCKED', False):
        raise PermissionError('The test split is locked until Section 11 (final evaluation).')
    if split not in _IMAGE_CACHE:
        sub = split_df[split_df['split'] == split].sort_values('id_code')
        with ThreadPoolExecutor(CFG.io_threads) as pool:
            imgs = list(pool.map(lambda i: read_rgb(LOCAL_CACHE / f'{i}.png'), sub['id_code']))
        _IMAGE_CACHE[split] = (np.stack(imgs), sub[CFG.label_col].to_numpy(), sub['id_code'].tolist())
    return _IMAGE_CACHE[split]

for s in ('train', 'val1', 'val2'):
    x, y, _ = load_split(s)
    print(f'{s:5s}: {len(y):5d} images | DR fraction {y.mean():.3f} | array {x.shape} {x.nbytes / 1e9:.2f} GB')

fig, ax = plt.subplots(figsize=(7, 3))
pd.crosstab(split_df['split'], split_df['diagnosis'], normalize='index').reindex(['train', 'val1', 'val2', 'test']).plot(
    kind='bar', stacked=True, ax=ax, colormap='viridis'); ax.set_ylabel('fraction'); ax.set_title('Grade mix per split')
savefig(fig, 'p0_split_grades.png')

## Section 3 — Data pipeline: augmentation and datasets

In [ ]:
%%shared augment
# Section 3 — Data pipeline: augmentation and datasets  [shared with the app: augment]
PAPER_AUG = dict(rotation=30.0, shift=0.2, shear_deg=0.2, zoom=0.7, brightness=(0.5, 1.5), hflip=True, vflip=True,
                 fill='reflect')
MILD_AUG = dict(rotation=15.0, shift=0.05, shear_deg=0.2, zoom=0.1, brightness=(0.85, 1.15), hflip=True, vflip=True,
                fill='constant')
AUG_SETS = {'paper': PAPER_AUG, 'mild': MILD_AUG, 'none': None}

def random_augment(rgb, rng, params):
    if params is None:
        return rgb
    h, w = rgb.shape[:2]
    angle = rng.uniform(-params['rotation'], params['rotation'])
    zoom = rng.uniform(1 - params['zoom'], 1 + params['zoom'])
    shear = math.radians(rng.uniform(-params['shear_deg'], params['shear_deg']))
    tx = rng.uniform(-params['shift'], params['shift']) * w
    ty = rng.uniform(-params['shift'], params['shift']) * h
    c = np.array([[1, 0, -w / 2], [0, 1, -h / 2], [0, 0, 1]], dtype=np.float64)
    sh = np.array([[1, -math.sin(shear), 0], [0, math.cos(shear), 0], [0, 0, 1]], dtype=np.float64)
    rs = np.vstack([cv2.getRotationMatrix2D((0, 0), angle, 1.0 / zoom), [0, 0, 1]])
    back = np.array([[1, 0, w / 2 + tx], [0, 1, h / 2 + ty], [0, 0, 1]], dtype=np.float64)
    m = (back @ rs @ sh @ c)[:2]
    border = cv2.BORDER_REFLECT if params['fill'] == 'reflect' else cv2.BORDER_CONSTANT
    out = cv2.warpAffine(rgb, m, (w, h), flags=cv2.INTER_LINEAR, borderMode=border, borderValue=(0, 0, 0))
    if params['hflip'] and rng.random() < 0.5:
        out = out[:, ::-1]
    if params['vflip'] and rng.random() < 0.5:
        out = out[::-1]
    b = rng.uniform(*params['brightness'])
    return np.clip(out.astype(np.float32) * b, 0, 255).astype(np.uint8)

In [ ]:
%%shared pipeline
# Section 3 — Data pipeline: augmentation and datasets (continued)  [shared with the app: pipeline]
def prepare_input(rgb, input_type, afep_params=None, size=MODEL_INPUT_SIZE):
    img = cv2.resize(np.ascontiguousarray(rgb), (size, size), interpolation=cv2.INTER_AREA)
    if input_type == 'rgb':
        return img
    if input_type == 'afep_fused':
        return afep_fused(img, afep_params)
    if input_type == 'afep_binary':
        return afep_binary(img, afep_params)
    raise ValueError(f'unknown input_type {input_type}')

def to_tensor(img):
    x = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))

def make_model_tensor(rgb320, input_type, afep_params=None, aug_params=None, rng=None):
    if aug_params is not None:
        rgb320 = random_augment(rgb320, rng, aug_params)
    return to_tensor(prepare_input(rgb320, input_type, afep_params))

### 3.1 PyTorch datasets

In [ ]:
# 3.1 PyTorch datasets
class TrainEvalDataset(Dataset):
    def __init__(self, images, labels, input_type, afep_params=None, aug_params=None):
        self.images, self.labels = images, labels
        self.input_type, self.afep_params, self.aug_params = input_type, afep_params, aug_params

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        rng = np.random.default_rng(int(torch.randint(0, 2**31 - 1, (1,)).item())) if self.aug_params else None
        x = make_model_tensor(self.images[i], self.input_type, self.afep_params, self.aug_params, rng)
        return x, int(self.labels[i])

class TTDADataset(Dataset):
    def __init__(self, images, n_aug, input_type, afep_params, aug_params, seed):
        self.images, self.n_aug, self.input_type = images, n_aug, input_type
        self.afep_params, self.aug_params, self.seed = afep_params, aug_params, seed

    def __len__(self):
        return len(self.images) * self.n_aug

    def __getitem__(self, k):
        i, j = divmod(k, self.n_aug)
        rng = np.random.default_rng([self.seed, i, j])
        return make_model_tensor(self.images[i], self.input_type, self.afep_params, self.aug_params, rng), i, j

def make_loader(ds, shuffle, seed=0, batch_size=None):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size or CFG.batch_size, shuffle=shuffle, num_workers=CFG.num_workers,
                      pin_memory=DEVICE.type == 'cuda', generator=g, persistent_workers=False)

x_tr, y_tr, ids_tr = load_split('train')
rng = np.random.default_rng(0)
ims = [x_tr[0]] + [random_augment(x_tr[0], rng, PAPER_AUG) for _ in range(5)] + \
      [random_augment(x_tr[0], rng, MILD_AUG) for _ in range(2)]
show_grid(ims, ['original'] + ['paper aug'] * 5 + ['mild aug'] * 2, ncols=4, cell=2.4,
          title='Augmentation examples', save='p0_augmentation_examples.png')

## Section 4 — P1: model, training engine, metrics, baseline "Model 1"

In [ ]:
%%shared model
# Section 4 — P1: model, training engine, metrics, baseline "Model 1"  [shared with the app: model]
def swap_activations(module, old=nn.SiLU, new=nn.Mish):
    n = 0
    for name, child in module.named_children():
        if isinstance(child, old):
            setattr(module, name, new()); n += 1
        else:
            n += swap_activations(child, old, new)
    return n

def build_model(activation='silu', head_hidden=512, dropout=0.2, pretrained=True):
    weights = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.efficientnet_b0(weights=weights)
    act = nn.Mish if activation == 'mish' else nn.SiLU
    model.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.classifier[1].in_features, head_hidden),
                                     act(), nn.Linear(head_hidden, 2))
    if activation == 'mish':
        swap_activations(model)
    return model

In [ ]:
# 4.1b Check the model
def count_modules(model, kind):
    return sum(isinstance(m, kind) for m in model.modules())

_m = build_model('mish')
print('parameters (millions):', round(sum(p.numel() for p in _m.parameters()) / 1e6, 2))
print('Mish model -> SiLU left:', count_modules(_m, nn.SiLU), '| Mish modules:', count_modules(_m, nn.Mish))
assert count_modules(_m, nn.SiLU) == 0
print('SiLU model -> SiLU modules:', count_modules(build_model('silu'), nn.SiLU))
del _m

### 4.2 Metrics (what each number means)

In [ ]:
# 4.2 Metrics (what each number means)
def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def softmax_np(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def ece_score(probs, y, n_bins=15):
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def compute_metrics(y, probs, n_bins=None):
    y = np.asarray(y).astype(int); pred = probs.argmax(1); p1 = probs[:, 1]
    tp, tn = int(((pred == 1) & (y == 1)).sum()), int(((pred == 0) & (y == 0)).sum())
    fp, fn = int(((pred == 1) & (y == 0)).sum()), int(((pred == 0) & (y == 1)).sum())
    div = lambda a, b: a / b if b else float('nan')
    sens, prec = div(tp, tp + fn), div(tp, tp + fp)
    return {'n': len(y), 'accuracy': div(tp + tn, len(y)), 'sensitivity': sens, 'specificity': div(tn, tn + fp),
            'precision': prec, 'f1': div(2 * prec * sens, prec + sens),
            'auc': float(roc_auc_score(y, p1)) if len(np.unique(y)) == 2 else float('nan'),
            'nll': float(-np.mean(np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1)))),
            'brier': float(np.mean((p1 - y) ** 2)), 'ece': ece_score(probs, y, n_bins or CFG.ece_bins),
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

CI_KEYS = ('accuracy', 'sensitivity', 'specificity', 'f1', 'auc', 'ece', 'nll', 'brier')

def metrics_with_ci(y, probs, n_boot=None, seed=0):
    point = compute_metrics(y, probs)
    rng, n_boot = np.random.default_rng(seed), n_boot or CFG.n_bootstrap
    samples = {k: [] for k in CI_KEYS}
    y = np.asarray(y)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        for _ in range(n_boot):
            idx = rng.integers(0, len(y), len(y))
            if len(np.unique(y[idx])) < 2:
                continue
            m = compute_metrics(y[idx], probs[idx])
            for k in CI_KEYS:
                samples[k].append(m[k])
        point['ci95'] = {k: ([float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] if len(v) else [float('nan')] * 2)
                         for k, v in samples.items()}
    return point

def mcnemar_test(y, pred_a, pred_b):
    a_ok, b_ok = pred_a == y, pred_b == y
    b, c = int((a_ok & ~b_ok).sum()), int((~a_ok & b_ok).sum())
    p = float(stats.binomtest(b, b + c, 0.5).pvalue) if b + c else 1.0
    return {'only_A_correct': b, 'only_B_correct': c, 'p_value': p}

def paired_bootstrap_diff(y, probs_a, probs_b, key, n_boot=None, seed=0):
    rng, n_boot, y = np.random.default_rng(seed), n_boot or CFG.n_bootstrap, np.asarray(y)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        diffs.append(compute_metrics(y[idx], probs_b[idx])[key] - compute_metrics(y[idx], probs_a[idx])[key])
    d = compute_metrics(y, probs_b)[key] - compute_metrics(y, probs_a)[key]
    lo, hi = np.percentile(diffs, [2.5, 97.5]) if diffs else (float('nan'), float('nan'))
    return {'metric': key, 'diff_B_minus_A': float(d), 'ci95': [float(lo), float(hi)],
            'ci_excludes_zero': bool(lo > 0 or hi < 0)}

def reliability_curve(probs, y, n_bins=None):
    n_bins = n_bins or CFG.ece_bins
    conf, correct = probs.max(1), probs.argmax(1) == y
    edges = np.linspace(0, 1, n_bins + 1); rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            rows.append((conf[m].mean(), correct[m].mean(), m.sum()))
    return np.array(rows)

_y = np.array([1, 0, 1, 0]); _p = np.array([[0.1, 0.9], [0.8, 0.2], [0.6, 0.4], [0.3, 0.7]])
_m = compute_metrics(_y, _p)
assert _m['tp'] == 1 and _m['tn'] == 1 and _m['fp'] == 1 and _m['fn'] == 1 and abs(_m['accuracy'] - 0.5) < 1e-9
assert abs(ece_score(np.array([[0.2, 0.8]] * 10), np.array([1] * 8 + [0] * 2))) < 1e-9
print('metric self-tests passed')

### 4.3 Training engine (resumable)

In [ ]:
# 4.3 Training engine (resumable)
def configure_trainable(model, unfreeze_from_stage):
    for p in model.parameters():
        p.requires_grad = False
    for i, stage in enumerate(model.features):
        if i >= unfreeze_from_stage:
            for p in stage.parameters():
                p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

def set_train_mode(model, unfreeze_from_stage):
    model.train()
    for i, stage in enumerate(model.features):
        if i < unfreeze_from_stage:
            stage.eval()

def make_optimizer(model):
    groups = [{'params': list(model.classifier.parameters()), 'lr': CFG.lr_head}]
    backbone = [p for p in model.features.parameters() if p.requires_grad]
    if backbone:
        groups.append({'params': backbone, 'lr': CFG.lr_backbone})
    return torch.optim.Adam(groups, weight_decay=CFG.weight_decay)

@torch.no_grad()
def predict_logits(model, images, input_type, afep_params=None, batch_size=64):
    ds = TrainEvalDataset(images, np.zeros(len(images), dtype=int), input_type, afep_params)
    model.eval(); out = []
    for x, _ in make_loader(ds, shuffle=False, batch_size=batch_size):
        with amp_ctx():
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy()

def _canon(obj):
    return json.loads(json.dumps(obj, default=_json_default))

def _atomic_torch_save(obj, path):
    tmp = Path(str(path) + '.tmp'); torch.save(obj, tmp); os.replace(tmp, path)

def save_predictions(run_dir, split, logits, y, ids):
    pd.DataFrame({'id_code': ids, 'label': y, 'logit_0': logits[:, 0], 'logit_1': logits[:, 1]}).to_csv(
        Path(run_dir) / f'predictions_{split}.csv', index=False)

def load_predictions(run_dir, split):
    df = pd.read_csv(Path(run_dir) / f'predictions_{split}.csv', dtype={'id_code': str})
    return df[['logit_0', 'logit_1']].to_numpy(), df['label'].to_numpy(), df['id_code'].tolist()

def load_model(run_dir, device=None):
    spec = load_json(Path(run_dir) / 'config.json')['spec']
    model = build_model(spec['activation'], CFG.head_hidden, CFG.dropout, pretrained=False)
    model.load_state_dict(torch.load(Path(run_dir) / 'best.pt', map_location='cpu', weights_only=False)['model'])
    return model.to(device or DEVICE).eval(), spec

def train_run(spec, afep_params=None):
    run_dir = RUNS_DIR / spec['name']
    if (run_dir / 'done.json').exists():
        print(f"✔ {spec['name']} already finished - skipped"); return run_dir
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / 'config.json').exists():
        old = load_json(run_dir / 'config.json')
        if old['spec'] != _canon(spec) or old['afep_params'] != _canon(afep_params):
            raise RuntimeError(f'{run_dir} exists with DIFFERENT settings. Use a new run name.')
    else:
        save_json({'spec': spec, 'afep_params': afep_params, 'config': CFG.to_dict(),
                   'environment': environment_info()}, run_dir / 'config.json')
    set_seed(spec['seed'])
    x_tr, y_tr, _ = load_split('train'); x_v, y_v, ids_v = load_split('val1')
    model = build_model(spec['activation'], CFG.head_hidden, CFG.dropout).to(DEVICE)
    configure_trainable(model, spec['unfreeze_from_stage'])
    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    start, best_loss, bad, history = 0, float('inf'), 0, []
    if (run_dir / 'last.pt').exists():
        st = torch.load(run_dir / 'last.pt', map_location=DEVICE, weights_only=False)
        model.load_state_dict(st['model']); opt.load_state_dict(st['optimizer']); scaler.load_state_dict(st['scaler'])
        start, best_loss, bad, history = st['epoch'] + 1, st['best_loss'], st['bad_epochs'], st['history']
        print(f'↻ resuming {spec["name"]} after epoch {start}')
    train_ds = TrainEvalDataset(x_tr, y_tr, spec['input_type'], afep_params, AUG_SETS[spec['aug']])
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(start, CFG.epochs):
        if bad >= CFG.patience:
            break
        t0 = time.time(); set_train_mode(model, spec['unfreeze_from_stage']); tot, n = 0.0, 0
        loader = make_loader(train_ds, shuffle=True, seed=spec['seed'] * 1000 + epoch)
        for b, (x, y) in enumerate(tqdm(loader, desc=f"{spec['name']} ep {epoch + 1}/{CFG.epochs}", leave=False)):
            if CFG.max_batches_per_epoch and b >= CFG.max_batches_per_epoch:
                break
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with amp_ctx():
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y); n += len(y)
        m = compute_metrics(y_v, softmax_np(predict_logits(model, x_v, spec['input_type'], afep_params)))
        row = {'epoch': epoch + 1, 'train_loss': tot / max(n, 1), 'val1_loss': m['nll'], 'val1_acc': m['accuracy'],
               'val1_auc': m['auc'], 'seconds': round(time.time() - t0, 1)}
        history.append(row)
        print(' | '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in row.items()))
        if m['nll'] < best_loss - 1e-4:
            best_loss, bad = m['nll'], 0
            _atomic_torch_save({'model': model.state_dict(), 'epoch': epoch + 1, 'val1_loss': best_loss}, run_dir / 'best.pt')
        else:
            bad += 1
        _atomic_torch_save({'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scaler': scaler.state_dict(),
                            'epoch': epoch, 'best_loss': best_loss, 'bad_epochs': bad, 'history': history}, run_dir / 'last.pt')
        pd.DataFrame(history).to_csv(run_dir / 'train_log.csv', index=False)
    model, _ = load_model(run_dir)
    for split in ('val1', 'val2'):
        x_s, y_s, ids_s = load_split(split)
        save_predictions(run_dir, split, predict_logits(model, x_s, spec['input_type'], afep_params), y_s, ids_s)
    logits, y1, _ = load_predictions(run_dir, 'val1')
    save_json(metrics_with_ci(y1, softmax_np(logits)), run_dir / 'metrics_val1.json')
    best_epoch = int(torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)['epoch'])
    save_json({'best_epoch': best_epoch, 'epochs_run': len(history),
               'train_minutes': round(sum(h['seconds'] for h in history) / 60, 2), 'finished': datetime.now().isoformat()},
              run_dir / 'done.json')
    (run_dir / 'last.pt').unlink(missing_ok=True)
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    print(f'✔ finished {spec["name"]} (best epoch {best_epoch})')
    return run_dir

def plot_history(run_dir, title):
    h = pd.read_csv(Path(run_dir) / 'train_log.csv')
    fig, ax = plt.subplots(1, 2, figsize=(9, 3))
    ax[0].plot(h['epoch'], h['train_loss'], label='train'); ax[0].plot(h['epoch'], h['val1_loss'], label='val1')
    ax[0].set_title('loss'); ax[0].legend(); ax[1].plot(h['epoch'], h['val1_auc']); ax[1].set_title('val1 ROC-AUC')
    fig.suptitle(title); savefig(fig, f'{Path(run_dir).name}_history.png')
print('training engine ready')

### 4.4 P1 — baseline "Model 1" (RGB input, SiLU, no augmentation)

In [ ]:
# 4.4 P1 — baseline "Model 1" (RGB input, SiLU, no augmentation)
P1_SPEC = dict(name='p1_model1_rgb_silu_noaug_s42', input_type='rgb', activation='silu', aug='none', seed=CFG.seed,
               unfreeze_from_stage=CFG.unfreeze_from_stage)
P1_DIR = train_run(P1_SPEC)
plot_history(P1_DIR, 'P1 baseline (Model 1)')
m = load_json(P1_DIR / 'metrics_val1.json')
print({k: round(m[k], 4) for k in ('accuracy', 'sensitivity', 'specificity', 'f1', 'auc', 'ece', 'nll')})
print('95% CI accuracy:', [round(v, 4) for v in m['ci95']['accuracy']], '| AUC:', [round(v, 4) for v in m['ci95']['auc']])

## Section 5 — P2: AFEP (Adaptive Fundus Enhancement Pipeline)

In [ ]:
%%shared afep
# Section 5 — P2: AFEP (Adaptive Fundus Enhancement Pipeline)  [shared with the app: afep]
AFEP_DEFAULTS = dict(channel='gray', median_ksize=3, variance_ksize=5, mask_threshold=10, scale_percentile=99.5,
                     suppress_rim=True, rim_width=7, otsu_source='fused_gray',
                     morph=[['open', 3], ['close', 5], ['close', 11]])

def _scale_u8(x, mask, pct):
    vals = x[mask] if mask.any() else x.ravel()
    hi = max(float(np.percentile(vals, pct)), 1e-6)
    return np.clip(x / hi * 255.0, 0, 255).astype(np.uint8)

def afep_layers(rgb, params=None):
    p = {**AFEP_DEFAULTS, **(params or {})}
    g = rgb[:, :, 1].copy() if p['channel'] == 'green' else cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    mask = g > p['mask_threshold']
    if p['suppress_rim']:
        k_rim = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * p['rim_width'] + 1, 2 * p['rim_width'] + 1))
        mask = cv2.erode(mask.astype(np.uint8), k_rim).astype(bool)
    med = cv2.medianBlur(g, p['median_ksize'])
    gf = g.astype(np.float32)
    kx = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype=np.float32)
    gx, gy = cv2.filter2D(gf, cv2.CV_32F, kx), cv2.filter2D(gf, cv2.CV_32F, kx.T)
    prewitt = _scale_u8(np.sqrt(gx ** 2 + gy ** 2), mask, p['scale_percentile']) * mask.astype(np.uint8)
    k = p['variance_ksize']
    mean, mean_sq = cv2.blur(gf, (k, k)), cv2.blur(gf * gf, (k, k))
    local_std = _scale_u8(np.sqrt(np.maximum(mean_sq - mean * mean, 0)), mask, p['scale_percentile']) * mask.astype(np.uint8)
    fused = np.dstack([med, prewitt, local_std])
    src = med if p['otsu_source'] == 'median' else cv2.cvtColor(fused, cv2.COLOR_RGB2GRAY)
    vals = src[mask] if mask.any() else src.ravel()
    t, _ = cv2.threshold(vals.reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    otsu = ((src > t) & mask).astype(np.uint8) * 255
    morph = otsu.copy()
    for op, size in p['morph']:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(size), int(size)))
        morph = cv2.morphologyEx(morph, cv2.MORPH_OPEN if op == 'open' else cv2.MORPH_CLOSE, kernel)
    return {'gray': g, 'median': med, 'prewitt': prewitt, 'variance': local_std, 'fused': fused,
            'otsu': otsu, 'morph': morph, 'otsu_threshold': float(t)}

def afep_fused(rgb, params=None):
    return afep_layers(rgb, params)['fused']

def afep_binary(rgb, params=None):
    return np.dstack([afep_layers(rgb, params)['morph']] * 3)

### 5.1 Look at AFEP (training images only) — compare with paper Fig. 3 / Fig. 4

In [ ]:
# 5.1 Look at AFEP (training images only) — compare with paper Fig. 3 / Fig. 4
x_tr, y_tr, ids_tr = load_split('train')
pick = np.random.default_rng(1).choice(len(x_tr), 3, replace=False)
steps = ['median', 'prewitt', 'variance', 'fused', 'otsu', 'morph']
ims, tts = [], []
for i in pick:
    img = prepare_input(x_tr[i], 'rgb'); L = afep_layers(img)
    ims += [img] + [L[s] for s in steps]; tts += [f'{ids_tr[i]} RGB (label {y_tr[i]})'] + steps
show_grid(ims, tts, ncols=7, cell=2.2, title='AFEP steps (like paper Fig. 3)', save='p2_afep_steps.png')

variants = {'Otsu(fused) open3+close5+close11 (default)': {},
            'Otsu(median) open3+close5+close11': {'otsu_source': 'median'},
            'Otsu(fused) open 3,5,11 (literal "erosion then dilation")': {'morph': [['open', 3], ['open', 5], ['open', 11]]},
            'Otsu(fused) green channel': {'channel': 'green'},
            'rim kept (paper-figure-like)': {'suppress_rim': False}}
ims, tts = [], []
for i in pick:
    img = prepare_input(x_tr[i], 'rgb')
    for name, v in variants.items():
        ims.append(afep_layers(img, v)['morph']); tts.append(name)
show_grid(ims, tts, ncols=5, cell=2.4, title='Binary-map variants', save='p2_afep_variants.png')

### 5.2 Confirm the AFEP settings before Section 6

In [ ]:
# 5.2 Confirm the AFEP settings before Section 6
AFEP_CHOICE = dict(AFEP_DEFAULTS)
AFEP_CHOICE_FILE = RESULTS_DIR / 'p2_afep_choice.json'
if AFEP_CHOICE_FILE.exists():
    AFEP_PARAMS = load_json(AFEP_CHOICE_FILE)['afep_params']
    if AFEP_PARAMS != _canon(AFEP_CHOICE):
        raise RuntimeError('AFEP settings are frozen (AFEP models exist) and differ from AFEP_CHOICE. '
                           'Restore the frozen values or start a new experiment version.')
    print('Using frozen AFEP settings:', AFEP_PARAMS)
else:
    AFEP_PARAMS = _canon(AFEP_CHOICE)
    print('AFEP settings (will be frozen when Section 6 starts):', AFEP_PARAMS)

_a = afep_fused(prepare_input(x_tr[0], 'rgb'), AFEP_PARAMS)
assert _a.shape == (224, 224, 3) and _a.dtype == np.uint8
assert np.array_equal(_a, afep_fused(prepare_input(x_tr[0], 'rgb'), AFEP_PARAMS))
assert set(np.unique(afep_binary(prepare_input(x_tr[0], 'rgb'), AFEP_PARAMS))) <= {0, 255}
print('AFEP checks passed')

### 5.3 AFEP cost: images per second of the CPU preprocessing (no model)

In [ ]:
# 5.3 AFEP cost: images per second of the CPU preprocessing (no model)
bench = {}
rng = np.random.default_rng(0)
for inp in ['rgb', 'afep_fused', 'afep_binary']:
    t0 = time.perf_counter(); n = min(200, len(x_tr))
    for i in range(n):
        make_model_tensor(x_tr[i], inp, AFEP_PARAMS, PAPER_AUG, rng)
    bench[inp] = round(n / (time.perf_counter() - t0), 1)
print('images/second on ONE CPU core (augment + preprocess):', bench)
save_json({'images_per_second_one_core': bench, 'environment': environment_info()}, RESULTS_DIR / 'p2_afep_speed.json')

## Section 6 — P3: Mish and the ablation grid

In [ ]:
# Section 6 — P3: Mish and the ablation grid
def ablation_specs(unfreeze=None, seeds=None, cells=None):
    specs = []
    for inp in CFG.ablation_inputs:
        for act in CFG.ablation_activations:
            if cells and (inp, act) not in cells:
                continue
            for s in seeds or CFG.ablation_seeds:
                u = CFG.unfreeze_from_stage if unfreeze is None else unfreeze
                tag = '' if u == CFG.unfreeze_from_stage else f'_unfreeze{u}'
                specs.append(dict(name=f'p3_{inp}_{act}_paperaug{tag}_s{s}', input_type=inp, activation=act,
                                  aug='paper', seed=s, unfreeze_from_stage=u))
    return specs

def afep_for(spec):
    return AFEP_PARAMS if spec['input_type'].startswith('afep') else None

if not AFEP_CHOICE_FILE.exists():
    save_json({'afep_params': AFEP_PARAMS, 'frozen_at': datetime.now().isoformat()}, AFEP_CHOICE_FILE)
    print('AFEP settings frozen ->', AFEP_CHOICE_FILE)

def run_rows(specs):
    rows = []
    for s in specs:
        f = RUNS_DIR / s['name'] / 'metrics_val1.json'
        if f.exists():
            m = load_json(f)
            rows.append({**{k: s[k] for k in ('name', 'input_type', 'activation', 'seed', 'unfreeze_from_stage')},
                         **{k: m[k] for k in ('accuracy', 'sensitivity', 'specificity', 'f1', 'auc', 'ece', 'nll')},
                         'auc_ci95': m['ci95']['auc']})
    return pd.DataFrame(rows)

def missing_runs(specs):
    return [s['name'] for s in specs if not (RUNS_DIR / s['name'] / 'done.json').exists()]

GRID = ablation_specs()
print(f'Ablation grid: {len(GRID)} runs | already finished on Drive: {len(GRID) - len(missing_runs(GRID))}')

#### 6.0b Train the grid

In [ ]:
# 6.0b Train the grid
MAX_RUNS_THIS_SESSION = None
done_now = 0
for k, spec in enumerate(GRID, 1):
    if MAX_RUNS_THIS_SESSION is not None and done_now >= MAX_RUNS_THIS_SESSION:
        print('Stopping as requested; re-run this cell later to continue.'); break
    was_done = (RUNS_DIR / spec['name'] / 'done.json').exists()
    print(f'[{k}/{len(GRID)}]', spec['name'])
    train_run(spec, afep_for(spec))
    done_now += 0 if was_done else 1

### 6.1 Ablation summary (val1 only)

In [ ]:
# 6.1 Ablation summary (val1 only)
abl = run_rows(GRID)
missing = missing_runs(GRID)
print(f'finished runs: {len(abl)}/{len(GRID)}', '| missing:', missing if missing else 'none')
if len(abl):
    abl.to_csv(RESULTS_DIR / 'p3_ablation_runs.csv', index=False)
    agg = abl.groupby(['input_type', 'activation'])[['accuracy', 'sensitivity', 'specificity', 'auc', 'ece', 'nll']].agg(['mean', 'std', 'count'])
    agg.to_csv(RESULTS_DIR / 'p3_ablation_summary.csv'); display(agg.round(4))
    tests = []
    for inp in CFG.ablation_inputs:
        for s in CFG.ablation_seeds:
            a, b = RUNS_DIR / f'p3_{inp}_silu_paperaug_s{s}', RUNS_DIR / f'p3_{inp}_mish_paperaug_s{s}'
            if (a / 'done.json').exists() and (b / 'done.json').exists():
                la, y, _ = load_predictions(a, 'val1'); lb, _, _ = load_predictions(b, 'val1')
                tests.append({'input': inp, 'seed': s, **mcnemar_test(y, la.argmax(1), lb.argmax(1)),
                              **{f'auc_{k}': v for k, v in paired_bootstrap_diff(y, softmax_np(la), softmax_np(lb), 'auc').items() if k != 'metric'}})
    if tests:
        tests = pd.DataFrame(tests); tests.to_csv(RESULTS_DIR / 'p3_silu_vs_mish_tests.csv', index=False)
        print('SiLU (A) vs Mish (B) on val1:'); display(tests)
    fig, ax = plt.subplots(figsize=(7, 3))
    means = abl.groupby(['input_type', 'activation'])['auc'].agg(['mean', 'std']).reset_index()
    ax.bar([f'{r.input_type}\n{r.activation}' for r in means.itertuples()], means['mean'], yerr=means['std'].fillna(0), capsize=4)
    ax.set_ylabel('val1 ROC-AUC'); ax.set_ylim(max(0, means['mean'].min() - 0.1), 1); ax.set_title('Ablation (mean ± std over seeds)')
    savefig(fig, 'p3_ablation_auc.png')

### 6.2 Select the model (pre-registered rule) + full fine-tuning variant

In [ ]:
# 6.2 Select the model (pre-registered rule) + full fine-tuning variant
SELECTION_FILE = RESULTS_DIR / 'p3_selection.json'
abl, missing = run_rows(GRID), missing_runs(GRID)
if SELECTION_FILE.exists():
    SELECTION = load_json(SELECTION_FILE); print('Frozen selection:', SELECTION)
elif missing:
    SELECTION = None
    print('Selection postponed: finish all ablation runs first (re-run the grid cell).')
else:
    cells = abl.groupby(['input_type', 'activation']).agg(auc=('auc', 'mean'), nll=('nll', 'mean')).reset_index()
    best = cells.sort_values(['auc', 'nll'], ascending=[False, True]).iloc[0]
    base_spec = [s for s in GRID if s['input_type'] == best.input_type and s['activation'] == best.activation and s['seed'] == CFG.seed][0]
    ft_spec = ablation_specs(unfreeze=0, seeds=[CFG.seed], cells=[(best.input_type, best.activation)])[0]
    train_run(ft_spec, afep_for(ft_spec))
    ft = load_json(RUNS_DIR / ft_spec['name'] / 'metrics_val1.json'); base = load_json(RUNS_DIR / base_spec['name'] / 'metrics_val1.json')
    chosen = ft_spec if (ft['auc'], -ft['nll']) > (base['auc'], -base['nll']) else base_spec
    SELECTION = {'rule': 'max mean val1 AUC over seeds (tie: min NLL); then seed-42 run vs full fine-tuning on val1 AUC',
                 'best_cell': {'input_type': best.input_type, 'activation': best.activation,
                               'mean_val1_auc': float(best.auc), 'mean_val1_nll': float(best.nll)},
                 'seed42_partial_ft': {'run': base_spec['name'], 'val1_auc': base['auc']},
                 'seed42_full_ft': {'run': ft_spec['name'], 'val1_auc': ft['auc']},
                 'selected_run': chosen['name'], 'selected_unfreeze_from_stage': chosen['unfreeze_from_stage'],
                 'selected_at': datetime.now().isoformat()}
    save_json(SELECTION, SELECTION_FILE); print(json.dumps(SELECTION, indent=2))
if SELECTION:
    SEL_DIR = RUNS_DIR / SELECTION['selected_run']
    SEL_SPEC = load_json(SEL_DIR / 'config.json')['spec']
    SEL_AFEP = afep_for(SEL_SPEC)
    print('Selected model:', SEL_DIR.name)

## Section 7 — P4: temperature scaling (confidence calibration)

In [ ]:
# Section 7 — P4: temperature scaling (confidence calibration)
def fit_temperature(logits, y, max_iter=100):
    z = torch.tensor(logits, dtype=torch.float64); t = torch.tensor(y, dtype=torch.long)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=max_iter, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(z / log_t.exp(), t); loss.backward(); return loss
    opt.step(closure)
    return float(log_t.exp().item())

def calibrate_run(run_dir):
    f = Path(run_dir) / 'temperature.json'
    if f.exists():
        return load_json(f)
    l2, y2, _ = load_predictions(run_dir, 'val2'); l1, y1, _ = load_predictions(run_dir, 'val1')
    T = fit_temperature(l2, y2)
    assert np.array_equal(softmax_np(l1).argmax(1), softmax_np(l1, T).argmax(1)), 'argmax must not change'
    res = {'T': T}
    for name, (l, y) in {'val2_fit_set': (l2, y2), 'val1_heldout': (l1, y1)}.items():
        before, after = compute_metrics(y, softmax_np(l)), compute_metrics(y, softmax_np(l, T))
        res[name] = {k: {'before': before[k], 'after': after[k]} for k in ('ece', 'nll', 'brier', 'accuracy')}
    save_json(res, f)
    return res

def plot_reliability(ax, probs, y, label):
    rc = reliability_curve(probs, y)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    if len(rc):
        ax.plot(rc[:, 0], rc[:, 1], 'o-', label=f'{label} (ECE={ece_score(probs, y):.3f})')
    ax.set_xlabel('confidence'); ax.set_ylabel('accuracy'); ax.set_xlim(0.45, 1.0); ax.set_ylim(0.45, 1.0); ax.legend(fontsize=7)

if not SELECTION:
    print('Run Section 6 (ablation + selection) first.')
else:
    CAL_RUNS = [P1_DIR, SEL_DIR] + [RUNS_DIR / s['name'] for s in ablation_specs(
        unfreeze=SEL_SPEC['unfreeze_from_stage'], seeds=CFG.ablation_seeds,
        cells=[(SEL_SPEC['input_type'], SEL_SPEC['activation'])])]
    CAL_RUNS = list(dict.fromkeys(CAL_RUNS))
    rows = []
    for rd in CAL_RUNS:
        if not (rd / 'done.json').exists():
            print('not trained (skipped):', rd.name); continue
        c = calibrate_run(rd)
        rows.append({'run': rd.name, 'T': round(c['T'], 4),
                     'val1_ECE_before': c['val1_heldout']['ece']['before'], 'val1_ECE_after': c['val1_heldout']['ece']['after'],
                     'val1_NLL_before': c['val1_heldout']['nll']['before'], 'val1_NLL_after': c['val1_heldout']['nll']['after']})
    cal_table = pd.DataFrame(rows); cal_table.to_csv(RESULTS_DIR / 'p4_calibration.csv', index=False); display(cal_table.round(4))
    T_SEL, T_BASE = load_json(SEL_DIR / 'temperature.json')['T'], load_json(P1_DIR / 'temperature.json')['T']
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, rd, T, title in [(axes[0], P1_DIR, T_BASE, 'Model 1 (baseline)'), (axes[1], SEL_DIR, T_SEL, 'Selected model')]:
        l1, y1, _ = load_predictions(rd, 'val1')
        plot_reliability(ax, softmax_np(l1), y1, 'before'); plot_reliability(ax, softmax_np(l1, T), y1, f'after (T={T:.2f})')
        ax.set_title(f'{title} - val1 (held-out)')
    savefig(fig, 'p4_reliability_val1.png')

## Section 8 — P5: Test-Time Data Augmentation (TTDA) and uncertainty

In [ ]:
%%shared uncertainty
# Section 8 — P5: Test-Time Data Augmentation (TTDA) and uncertainty  [shared with the app: uncertainty]
def entropy_bits(p, axis=-1):
    p = np.clip(p, 1e-12, 1.0)
    return -(p * np.log2(p)).sum(axis=axis)

def ttda_scores(probs_aug):
    mean_p = probs_aug.mean(axis=1)
    pred_ent = entropy_bits(mean_p)
    exp_ent = entropy_bits(probs_aug).mean(axis=1)
    return mean_p, pred_ent, exp_ent, pred_ent - exp_ent

@torch.no_grad()
def model_probs(model, x, T, device):
    model.eval()
    return torch.softmax(model(x.to(device)).float() / T, dim=1).cpu().numpy()

def ttda_predict(model, rgb320, n_aug, input_type, afep_params, aug_params, T, device, seed=0, batch=64):
    xs = torch.stack([make_model_tensor(rgb320, input_type, afep_params, aug_params, np.random.default_rng([seed, 0, j]))
                      for j in range(n_aug)])
    probs = np.concatenate([model_probs(model, xs[k:k + batch], T, device) for k in range(0, n_aug, batch)])
    mean_p, pe, ee, mi = ttda_scores(probs[None])
    return {'p_final': mean_p[0], 'uncertainty': float(pe[0]), 'expected_entropy': float(ee[0]),
            'mutual_info': float(mi[0]), 'n_forward': n_aug, 'used_ttda': True}

def selective_predict(model, rgb320, input_type, afep_params, T, tau, n_small, aug_params, device, seed=0):
    p = model_probs(model, make_model_tensor(rgb320, input_type, afep_params).unsqueeze(0), T, device)[0]
    if p.max() >= tau:
        return {'p_final': p, 'p_single': p, 'uncertainty': float(entropy_bits(p)), 'n_forward': 1, 'used_ttda': False}
    out = ttda_predict(model, rgb320, n_small, input_type, afep_params, aug_params, T, device, seed)
    out.update(p_single=p, n_forward=1 + n_small)
    return out

In [ ]:
# Section 8 — P5: Test-Time Data Augmentation (TTDA) and uncertainty (continued)
def run_ttda(run_dir, split, aug_set, n_aug=None, seed=None):
    n_aug, seed = n_aug or CFG.ttda_n_max, CFG.seed if seed is None else seed
    out = Path(run_dir) / 'ttda' / f'{split}_{aug_set}_n{n_aug}.npz'
    if out.exists():
        return np.load(out)['logits']
    out.parent.mkdir(exist_ok=True)
    model, spec = load_model(run_dir)
    x, y, ids = load_split(split)
    ds = TTDADataset(x, n_aug, spec['input_type'], afep_for(spec), AUG_SETS[aug_set], seed)
    logits = np.zeros((len(x), n_aug, 2), dtype=np.float32)
    with torch.no_grad():
        for xb, i, j in tqdm(make_loader(ds, shuffle=False, batch_size=CFG.ttda_batch), desc=f'TTDA {split} {aug_set} N={n_aug}'):
            with amp_ctx():
                logits[i.numpy(), j.numpy()] = model(xb.to(DEVICE, non_blocking=True)).float().cpu().numpy()
    np.savez_compressed(out, logits=logits, labels=y, ids=np.array(ids))
    del model
    return logits

def risk_coverage(y, pred, unc):
    order = np.argsort(unc, kind='stable')
    correct = (pred[order] == y[order]).astype(float)
    k = np.arange(1, len(y) + 1)
    return k / len(y), 1 - np.cumsum(correct) / k, float(np.mean(1 - np.cumsum(correct) / k))

if SELECTION:
    x2, y2, _ = load_split('val2'); l2_single, _, _ = load_predictions(SEL_DIR, 'val2')
    p_single2 = softmax_np(l2_single, T_SEL)
    rows, TTDA_VAL2 = [], {}
    rows.append({'aug_set': '-', 'N': 1, **{k: compute_metrics(y2, p_single2)[k] for k in ('accuracy', 'auc', 'ece', 'nll')},
                 'AURC_pred_entropy': risk_coverage(y2, p_single2.argmax(1), entropy_bits(p_single2))[2], 'spearman_vs_Nmax': np.nan})
    for aug_set in ['paper', 'mild']:
        L = run_ttda(SEL_DIR, 'val2', aug_set); TTDA_VAL2[aug_set] = L
        _, pe_full, _, _ = ttda_scores(softmax_np(L.reshape(-1, 2), T_SEL).reshape(L.shape))
        for n in CFG.ttda_n_grid:
            mean_p, pe, ee, mi = ttda_scores(softmax_np(L[:, :n].reshape(-1, 2), T_SEL).reshape(len(L), n, 2))
            m = compute_metrics(y2, mean_p)
            rows.append({'aug_set': aug_set, 'N': n, **{k: m[k] for k in ('accuracy', 'auc', 'ece', 'nll')},
                         'AURC_pred_entropy': risk_coverage(y2, mean_p.argmax(1), pe)[2],
                         'spearman_vs_Nmax': stats.spearmanr(pe, pe_full).statistic})
    ttda_table = pd.DataFrame(rows); ttda_table.to_csv(RESULTS_DIR / 'p5_ttda_val2.csv', index=False)
    print('val2, selected model, calibrated with T =', round(T_SEL, 3), '(N=1 is the single pass)'); display(ttda_table.round(4))

    TTDA_CHOICE_FILE = RESULTS_DIR / 'p5_ttda_choice.json'
    if TTDA_CHOICE_FILE.exists():
        TTDA_AUG = load_json(TTDA_CHOICE_FILE)['aug_set']
    else:
        full = ttda_table[ttda_table['N'] == CFG.ttda_n_max].sort_values('AURC_pred_entropy')
        TTDA_AUG = full.iloc[0]['aug_set']
        save_json({'aug_set': TTDA_AUG, 'rule': 'lower AURC (predictive entropy) on val2 at N_max',
                   'table': full.to_dict('records')}, TTDA_CHOICE_FILE)
    print('TTDA augmentation set used from now on:', TTDA_AUG)

    L = TTDA_VAL2[TTDA_AUG]
    mean_p, pe, ee, mi = ttda_scores(softmax_np(L.reshape(-1, 2), T_SEL).reshape(L.shape))
    correct = mean_p.argmax(1) == y2
    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    for ax, score, name in zip(axes, [pe, ee, mi], ['predictive entropy', 'expected entropy', 'mutual information']):
        ax.hist(score[correct], bins=30, alpha=0.6, density=True, label='correct')
        ax.hist(score[~correct], bins=30, alpha=0.6, density=True, label='wrong'); ax.set_title(name); ax.legend()
    fig.suptitle(f'val2: uncertainty of correct vs wrong predictions (TTDA {TTDA_AUG}, N={CFG.ttda_n_max})')
    savefig(fig, 'p5_uncertainty_correct_vs_wrong.png')

## Section 9 — P6: selective TTDA, referral and latency

In [ ]:
# Section 9 — P6: selective TTDA, referral and latency
def selective_from_logits(p_single, L, T, tau, n_small):
    gated = p_single.max(1) < tau
    ttda_p, pe, _, _ = ttda_scores(softmax_np(L[:, :n_small].reshape(-1, 2), T).reshape(len(L), n_small, 2))
    p = np.where(gated[:, None], ttda_p, p_single)
    unc = np.where(gated, pe, entropy_bits(p_single))
    return p, unc, gated

def uncertainty_strategies(p_single, L, T, tau, n_small):
    full_p, pe, ee, mi = ttda_scores(softmax_np(L.reshape(-1, 2), T).reshape(L.shape))
    sel_p, sel_unc, gated = selective_from_logits(p_single, L, T, tau, n_small)
    return {'single: 1 - max prob': (p_single, 1 - p_single.max(1)),
            'single: entropy': (p_single, entropy_bits(p_single)),
            'TTDA: predictive entropy': (full_p, pe), 'TTDA: expected entropy': (full_p, ee),
            'TTDA: mutual information': (full_p, mi), 'selective: entropy': (sel_p, sel_unc)}, gated

if SELECTION:
    L2 = TTDA_VAL2[TTDA_AUG]
    full_pred = ttda_scores(softmax_np(L2.reshape(-1, 2), T_SEL).reshape(L2.shape))[0].argmax(1)
    grid = []
    for tau in CFG.tau_grid:
        for n_small in CFG.n_small_grid:
            p, unc, gated = selective_from_logits(p_single2, L2, T_SEL, tau, n_small)
            grid.append({'tau': tau, 'n_small': n_small, 'fraction_ttda': gated.mean(),
                         'forward_passes_per_image': 1 + gated.mean() * n_small,
                         'agreement_with_full_ttda': (p.argmax(1) == full_pred).mean(),
                         'AURC': risk_coverage(y2, p.argmax(1), unc)[2], 'ECE': ece_score(p, y2)})
    grid = pd.DataFrame(grid); grid.to_csv(RESULTS_DIR / 'p6_selective_grid_val2.csv', index=False); display(grid.round(4))

    SELECTIVE_FILE = RESULTS_DIR / 'p6_selective_choice.json'
    if SELECTIVE_FILE.exists():
        SELECTIVE = load_json(SELECTIVE_FILE)
    else:
        ok_rows = grid[grid['agreement_with_full_ttda'] >= CFG.min_agreement]
        if ok_rows.empty:
            print('WARNING: no setting reaches the agreement target; taking the highest agreement.')
            ok_rows = grid.sort_values('agreement_with_full_ttda', ascending=False).head(1)
        ch = ok_rows.sort_values(['forward_passes_per_image', 'AURC']).iloc[0]
        strategies, _ = uncertainty_strategies(p_single2, L2, T_SEL, ch['tau'], int(ch['n_small']))
        thresholds = {k: float(np.quantile(u, 1 - CFG.referral_rate)) for k, (_, u) in strategies.items()}
        SELECTIVE = {'tau': float(ch['tau']), 'n_small': int(ch['n_small']), 'val2_row': ch.to_dict(),
                     'referral_rate': CFG.referral_rate, 'referral_thresholds_val2': thresholds,
                     'rule': f'agreement >= {CFG.min_agreement}, then fewest forward passes, then AURC'}
        save_json(SELECTIVE, SELECTIVE_FILE)
    print(json.dumps({k: SELECTIVE[k] for k in ('tau', 'n_small', 'val2_row')}, indent=2, default=_json_default))

    strategies, _ = uncertainty_strategies(p_single2, L2, T_SEL, SELECTIVE['tau'], SELECTIVE['n_small'])
    fig, ax = plt.subplots(figsize=(6, 4)); aurc_rows = []
    for name, (p, u) in strategies.items():
        cov, risk, aurc = risk_coverage(y2, p.argmax(1), u)
        ax.plot(cov, risk, label=f'{name} (AURC={aurc:.4f})'); aurc_rows.append({'strategy': name, 'AURC_val2': aurc})
    ax.axvline(1 - CFG.referral_rate, color='gray', ls=':'); ax.set_xlabel('coverage (share decided automatically)')
    ax.set_ylabel('risk (error rate)'); ax.legend(fontsize=7); ax.set_title('Risk-coverage on val2')
    savefig(fig, 'p6_risk_coverage_val2.png')
    pd.DataFrame(aurc_rows).to_csv(RESULTS_DIR / 'p6_aurc_val2.csv', index=False); display(pd.DataFrame(aurc_rows).round(5))

### 9.1 Latency benchmark (measured, end-to-end per image)

In [ ]:
# 9.1 Latency benchmark (measured, end-to-end per image)
def time_fn(fn, device, repeats=1):
    times = []
    for _ in range(repeats):
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); fn()
        if device.type == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times))

LATENCY_FILE = RESULTS_DIR / 'p6_latency.json'
if SELECTION and not LATENCY_FILE.exists():
    lat = {'environment': environment_info(), 'cpu_threads': torch.get_num_threads(), 'results': {}}
    for dev, n_img in [(DEVICE, CFG.latency_images_gpu), (torch.device('cpu'), CFG.latency_images_cpu)]:
        if dev.type == 'cpu' and DEVICE.type == 'cpu' and 'cpu' in lat['results']:
            continue
        model, spec = load_model(SEL_DIR, dev); afp = afep_for(spec); aug = AUG_SETS[TTDA_AUG]
        imgs = x2[:n_img]
        single = lambda im: model_probs(model, make_model_tensor(im, spec['input_type'], afp).unsqueeze(0), T_SEL, dev)
        full = lambda im: ttda_predict(model, im, CFG.ttda_n_max, spec['input_type'], afp, aug, T_SEL, dev)
        sel = lambda im: selective_predict(model, im, spec['input_type'], afp, T_SEL, SELECTIVE['tau'], SELECTIVE['n_small'], aug, dev)
        for im in imgs[:2]:
            single(im); sel(im)
        res = {}
        for name, fn in [('single_pass', single), ('full_ttda', full), ('selective_ttda', sel)]:
            ms = [time_fn(lambda: fn(im), dev) for im in tqdm(imgs, desc=f'{name} on {dev.type}', leave=False)]
            res[name] = {'mean_ms': float(np.mean(ms)), 'median_ms': float(np.median(ms)), 'p95_ms': float(np.percentile(ms, 95)), 'n_images': len(ms)}
        res['selective_fraction_ttda_in_sample'] = float(np.mean([sel(im)['used_ttda'] for im in imgs]))
        res['latency_reduction_selective_vs_full_mean'] = 1 - res['selective_ttda']['mean_ms'] / res['full_ttda']['mean_ms']
        lat['results'][dev.type] = res
        del model
    save_json(lat, LATENCY_FILE)
if LATENCY_FILE.exists():
    print(json.dumps(load_json(LATENCY_FILE)['results'], indent=2))

## Section 10 — P7: Grad-CAM (where does the model look?)

In [ ]:
%%shared gradcam
# Section 10 — P7: Grad-CAM (where does the model look?)  [shared with the app: gradcam]
def grad_cam(model, x, target_class=None):
    store = {}
    def fwd_hook(module, inp, out):
        store['act'] = out
        out.register_hook(lambda g: store.__setitem__('grad', g))
    handle = model.features[-1].register_forward_hook(fwd_hook)
    try:
        model.eval()
        with torch.enable_grad():
            x = x.clone().requires_grad_(True)
            logits = model(x)
            cls = int(logits.argmax(1)) if target_class is None else int(target_class)
            model.zero_grad(set_to_none=True)
            logits[0, cls].backward()
        act, grad = store['act'][0].detach(), store['grad'][0].detach()
        cam = F.relu((grad.mean(dim=(1, 2))[:, None, None] * act).sum(0))
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
        return (cam / (cam.max() + 1e-8)).cpu().numpy(), cls
    finally:
        handle.remove()

def cam_overlay(rgb, cam, alpha=0.45):
    rgb = cv2.resize(np.ascontiguousarray(rgb), (cam.shape[1], cam.shape[0]), interpolation=cv2.INTER_AREA)
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return (rgb * (1 - alpha) + heat * alpha).astype(np.uint8)

In [ ]:
# Section 10 — P7: Grad-CAM (where does the model look?) (continued)
if SELECTION:
    model, spec = load_model(SEL_DIR); afp = afep_for(spec)
    L2 = TTDA_VAL2[TTDA_AUG]
    mean_p, pe, _, _ = ttda_scores(softmax_np(L2.reshape(-1, 2), T_SEL).reshape(L2.shape))
    pred = mean_p.argmax(1); correct = pred == y2
    q_lo, q_hi = np.quantile(pe, [0.25, 0.75])
    groups = {'confident & correct': np.where(correct & (pe <= q_lo))[0], 'uncertain & correct': np.where(correct & (pe >= q_hi))[0],
              'confident & WRONG': np.where(~correct & (pe <= np.median(pe)))[0], 'uncertain & WRONG': np.where(~correct & (pe > np.median(pe)))[0]}
    ims, tts = [], []
    for gname, idx in groups.items():
        for i in idx[:2]:
            x = make_model_tensor(x2[i], spec['input_type'], afp).unsqueeze(0).to(DEVICE)
            cam, cls = grad_cam(model, x)
            ims += [prepare_input(x2[i], 'rgb'), prepare_input(x2[i], spec['input_type'], afp), cam_overlay(prepare_input(x2[i], 'rgb'), cam)]
            tts += [f'{gname} | label {y2[i]}', f'model input | P(DR)={mean_p[i, 1]:.2f}', f'Grad-CAM class {cls} | H={pe[i]:.2f} bits']
    show_grid(ims, tts, ncols=3, cell=2.8, title='Grad-CAM on val2 (selected model)', save='p7_gradcam_val2.png')

    torch.manual_seed(0)
    rand_model = build_model(spec['activation'], CFG.head_hidden, CFG.dropout, pretrained=False).to(DEVICE).eval()
    corrs = []
    for i in range(min(20, len(x2))):
        x = make_model_tensor(x2[i], spec['input_type'], afp).unsqueeze(0).to(DEVICE)
        c1, cls = grad_cam(model, x); c2, _ = grad_cam(rand_model, x, target_class=cls)
        corrs.append(stats.spearmanr(cv2.resize(c1, (28, 28)).ravel(), cv2.resize(c2, (28, 28)).ravel()).statistic)
    sanity = {'mean_spearman_trained_vs_random': float(np.nanmean(corrs)), 'n_images': len(corrs),
              'interpretation': 'low values mean the maps depend on the learned weights (sanity check passed)'}
    save_json(sanity, RESULTS_DIR / 'p7_gradcam_sanity.json'); print(sanity)
    del model, rand_model

## Section 11 — P8: final evaluation (test set + IDRiD) and statistics

In [ ]:
# Section 11 — P8: final evaluation (test set + IDRiD) and statistics
RUN_FINAL_TEST_EVALUATION = False or os.environ.get('TRUSTEYE_FINAL_EVAL') == '1'
FINAL_DIR = RESULTS_DIR / 'final'

def current_choices():
    return _canon({'baseline_run': P1_DIR.name, 'selected_run': SEL_DIR.name, 'selected_spec': SEL_SPEC,
                   'afep_params': SEL_AFEP, 'T_selected': T_SEL, 'T_baseline': T_BASE, 'ttda_aug_set': TTDA_AUG,
                   'ttda_n_max': CFG.ttda_n_max, 'tau': SELECTIVE['tau'], 'n_small': SELECTIVE['n_small'],
                   'referral_thresholds_val2': SELECTIVE['referral_thresholds_val2'],
                   'split_sha256': sha256_file(SPLIT_CSV), 'label_col': CFG.label_col})

if not RUN_FINAL_TEST_EVALUATION:
    print('Final evaluation not requested (RUN_FINAL_TEST_EVALUATION = False). The test set stays locked.')
elif not (SELECTION and LATENCY_FILE.exists()):
    print('Complete Sections 6-9 first.')
else:
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    frozen_file = FINAL_DIR / 'frozen_choices.json'
    choices = current_choices()
    if frozen_file.exists():
        if load_json(frozen_file)['choices'] != choices:
            raise RuntimeError('Choices differ from the frozen ones used for the test evaluation. Do not tune after testing.')
    else:
        save_json({'choices': choices, 'frozen_at': datetime.now().isoformat()}, frozen_file)
    FINAL_EVALUATION_UNLOCKED = True
    print('Test set unlocked. Frozen choices:', json.dumps(choices, indent=1, default=_json_default)[:1500])

In [ ]:
# Section 11 — P8: final evaluation (test set + IDRiD) and statistics (continued)
def evaluate_dataset(tag, images, y, ids):
    out = {}
    for name, rd, T in [('model1', P1_DIR, T_BASE), ('selected', SEL_DIR, T_SEL)]:
        f = rd / f'predictions_{tag}.csv'
        if not f.exists():
            model, spec = load_model(rd)
            save_predictions(rd, tag, predict_logits(model, images, spec['input_type'], afep_for(spec)), y, ids); del model
        logits, _, _ = load_predictions(rd, tag)
        out[f'{name}_raw'], out[f'{name}_calibrated'] = softmax_np(logits), softmax_np(logits, T)
    if tag == 'test':
        L = run_ttda(SEL_DIR, 'test', TTDA_AUG)
    else:
        _IMAGE_CACHE[tag] = (images, y, ids); L = run_ttda(SEL_DIR, tag, TTDA_AUG)
    strategies, gated = uncertainty_strategies(out['selected_calibrated'], L, T_SEL, SELECTIVE['tau'], SELECTIVE['n_small'])
    out['selected_full_ttda'] = strategies['TTDA: predictive entropy'][0]
    out['selected_selective_ttda'] = strategies['selective: entropy'][0]
    return out, strategies, gated

PAPER_REPORTED = pd.DataFrame([
    {'model': 'paper Model 1', 'accuracy': 0.9061, 'specificity': 0.86, 'sensitivity': 0.93, 'auc': 0.97, 'f1': 0.97, 'ece': 0.11},
    {'model': 'paper Model 2', 'accuracy': 0.93, 'specificity': 0.90, 'sensitivity': 0.95, 'auc': 0.97, 'f1': 0.97, 'ece': 0.11},
    {'model': 'paper Model 3 (proposed)', 'accuracy': 0.96, 'specificity': 0.95, 'sensitivity': 0.96, 'auc': 0.98, 'f1': 0.98, 'ece': 0.03}])

def report(tag, out, strategies, gated, y):
    rows = {k: metrics_with_ci(y, p) for k, p in out.items()}
    table = pd.DataFrame([{'row': k, **{m: v[m] for m in ('accuracy', 'sensitivity', 'specificity', 'f1', 'auc', 'ece', 'nll', 'brier')},
                           'auc_ci95': v['ci95']['auc'], 'ece_ci95': v['ci95']['ece']} for k, v in rows.items()])
    stats_tests = {
        'mcnemar_model1cal_vs_selective': mcnemar_test(y, out['model1_calibrated'].argmax(1), out['selected_selective_ttda'].argmax(1)),
        'auc_selective_minus_model1': paired_bootstrap_diff(y, out['model1_calibrated'], out['selected_selective_ttda'], 'auc'),
        'ece_selective_minus_model1raw': paired_bootstrap_diff(y, out['model1_raw'], out['selected_selective_ttda'], 'ece'),
        'ece_calibrated_minus_raw_selected': paired_bootstrap_diff(y, out['selected_raw'], out['selected_calibrated'], 'ece'),
        'auc_selective_minus_full_ttda': paired_bootstrap_diff(y, out['selected_full_ttda'], out['selected_selective_ttda'], 'auc'),
        'agreement_selective_vs_full': float((out['selected_selective_ttda'].argmax(1) == out['selected_full_ttda'].argmax(1)).mean()),
        'fraction_ttda': float(gated.mean()), 'forward_passes_per_image': float(1 + gated.mean() * SELECTIVE['n_small'])}
    referral, fig_rows = [], []
    fig, ax = plt.subplots(figsize=(6, 4))
    for name, (p, u) in strategies.items():
        cov, risk, aurc = risk_coverage(y, p.argmax(1), u)
        thr = SELECTIVE['referral_thresholds_val2'][name]; keep = u <= thr
        referral.append({'strategy': name, 'AURC': aurc, 'referred_fraction': float(1 - keep.mean()),
                         'accuracy_all': float((p.argmax(1) == y).mean()),
                         'accuracy_on_automated': float((p.argmax(1)[keep] == y[keep]).mean()) if keep.any() else np.nan})
        ax.plot(cov, risk, label=f'{name} ({aurc:.4f})')
    ax.set_xlabel('coverage'); ax.set_ylabel('risk'); ax.legend(fontsize=7); ax.set_title(f'Risk-coverage: {tag}')
    savefig(fig, f'p8_risk_coverage_{tag}.png')
    fig, ax = plt.subplots(figsize=(4.5, 4))
    plot_reliability(ax, out['selected_raw'], y, 'selected raw'); plot_reliability(ax, out['selected_calibrated'], y, 'selected calibrated')
    plot_reliability(ax, out['selected_selective_ttda'], y, 'selective TTDA'); ax.set_title(f'Reliability: {tag}')
    savefig(fig, f'p8_reliability_{tag}.png')
    referral = pd.DataFrame(referral)
    table.to_csv(FINAL_DIR / f'metrics_{tag}.csv', index=False); referral.to_csv(FINAL_DIR / f'referral_{tag}.csv', index=False)
    save_json({'rows': rows, 'statistics': stats_tests, 'referral': referral.to_dict('records'), 'n': len(y),
               'environment': environment_info()}, FINAL_DIR / f'metrics_{tag}.json')
    display(table.round(4)); display(referral.round(4)); print(json.dumps(stats_tests, indent=2, default=_json_default))

if RUN_FINAL_TEST_EVALUATION and globals().get('FINAL_EVALUATION_UNLOCKED'):
    xt, yt, idt = load_split('test')
    out, strategies, gated = evaluate_dataset('test', xt, yt, idt)
    report('test', out, strategies, gated, yt)
    seed_rows = []
    for s in CFG.ablation_seeds:
        rd = RUNS_DIR / SEL_DIR.name.replace(f'_s{CFG.seed}', f'_s{s}')
        if (rd / 'done.json').exists():
            if not (rd / 'predictions_test.csv').exists():
                model, spec = load_model(rd); save_predictions(rd, 'test', predict_logits(model, xt, spec['input_type'], afep_for(spec)), yt, idt); del model
            logits, _, _ = load_predictions(rd, 'test')
            seed_rows.append({'run': rd.name, **compute_metrics(yt, softmax_np(logits, calibrate_run(rd)['T']))})
    seeds = pd.DataFrame(seed_rows); seeds.to_csv(FINAL_DIR / 'test_seeds.csv', index=False)
    print('Selected cell over seeds (test, single pass, calibrated):')
    display(seeds[['accuracy', 'sensitivity', 'specificity', 'auc', 'ece']].agg(['mean', 'std']).round(4))
    print('Reported by Verma et al. (2026) on IDRiD - different data and protocol, for context only:'); display(PAPER_REPORTED)

In [ ]:
# 11.2 IDRiD external test (same frozen choices, used once)
if RUN_FINAL_TEST_EVALUATION and globals().get('FINAL_EVALUATION_UNLOCKED'):
    if not IDRID_ZIP.exists():
        print('IDRiD cache not available - external test skipped (see Section 2.9).')
    else:
        idrid_dir = restore_cache_from_drive(IDRID_ZIP)
        idl = pd.read_csv(IDRID_LABELS_CSV, dtype={'id_code': str}).sort_values('id_code')
        with ThreadPoolExecutor(CFG.io_threads) as pool:
            xi = np.stack(list(pool.map(lambda i: read_rgb(idrid_dir / f'{i}.png'), idl['id_code'])))
        yi = idl['label_binary'].to_numpy(); idi = idl['id_code'].tolist()
        out_i, strat_i, gated_i = evaluate_dataset('idrid', xi, yi, idi)
        report('idrid', out_i, strat_i, gated_i, yi)

## Section 12 — P9: Streamlit demo (CPU)

In [ ]:
# Section 12 — P9: Streamlit demo (CPU)
REQUIRED_SHARED = ['header', 'fundus', 'augment', 'pipeline', 'afep', 'model', 'uncertainty', 'gradcam']
if not SELECTION or not (RESULTS_DIR / 'p6_selective_choice.json').exists():
    print('Complete Sections 6-9 first.')
else:
    missing_shared = [k for k in REQUIRED_SHARED if k not in SHARED_CELLS]
    assert not missing_shared, f'Run these %%shared cells first in this session: {missing_shared}'
    src = ['# TrustEye-Edge shared code - GENERATED from TrustEye_Edge.ipynb (%%shared cells). Do not edit.']
    src += [f'# ===== {k} =====\n{SHARED_CELLS[k]}' for k in REQUIRED_SHARED]
    (DEPLOY_DIR / 'trusteye_shared.py').write_text('\n\n'.join(src), encoding='utf-8')
    torch.save(load_model(SEL_DIR, torch.device('cpu'))[0].state_dict(), DEPLOY_DIR / 'model_selected.pt')
    meta = {'run': SEL_DIR.name, 'activation': SEL_SPEC['activation'], 'input_type': SEL_SPEC['input_type'],
            'afep_params': SEL_AFEP, 'head_hidden': CFG.head_hidden, 'dropout': CFG.dropout, 'temperature': T_SEL,
            'tau': SELECTIVE['tau'], 'n_small': SELECTIVE['n_small'], 'ttda_aug_set': TTDA_AUG,
            'referral_threshold': SELECTIVE['referral_thresholds_val2']['selective: entropy'],
            'referral_rate_on_val2': CFG.referral_rate, 'label_scheme': '0 = No DR, 1 = DR (grades 1-4)',
            'exported_at': datetime.now().isoformat()}
    save_json(meta, DEPLOY_DIR / 'model_meta.json')
    print('Exported to', DEPLOY_DIR, ':', sorted(p.name for p in DEPLOY_DIR.iterdir()))

In [ ]:
%%writefile {DEPLOY_DIR}/app.py
# TrustEye-Edge Streamlit demo - generated by TrustEye_Edge.ipynb
import json
import time
from pathlib import Path

import cv2
import numpy as np
import streamlit as st
import torch

import trusteye_shared as ts

HERE = Path(__file__).resolve().parent
st.set_page_config(page_title='TrustEye-Edge demo', layout='wide')

@st.cache_resource
def load_bundle():
    meta = json.loads((HERE / 'model_meta.json').read_text())
    model = ts.build_model(meta['activation'], meta['head_hidden'], meta['dropout'], pretrained=False)
    model.load_state_dict(torch.load(HERE / 'model_selected.pt', map_location='cpu'))
    return model.eval(), meta

model, meta = load_bundle()
st.title('TrustEye-Edge - diabetic retinopathy triage (research prototype)')
with st.sidebar:
    device_name = st.radio('Run on', ['cpu'] + (['cuda'] if torch.cuda.is_available() else []))
    st.markdown(f"**Model:** `{meta['run']}`  \n**Input:** {meta['input_type']} | **activation:** {meta['activation']}  \n"
                f"**T** = {meta['temperature']:.3f} | **tau** = {meta['tau']} | **n_small** = {meta['n_small']}  \n"
                f"**Referral threshold** = {meta['referral_threshold']:.3f} bits (refers ~{meta['referral_rate_on_val2']:.0%} of val2)")
device = torch.device(device_name)
model.to(device)

upload = st.file_uploader('Upload a colour fundus photograph', type=['png', 'jpg', 'jpeg', 'tif', 'tiff'])
if upload is not None:
    raw = cv2.imdecode(np.frombuffer(upload.read(), np.uint8), cv2.IMREAD_UNCHANGED)
    if raw is None:
        st.error('This file could not be read as an image.')
        st.stop()
    t0 = time.perf_counter()
    bgr, _ = ts.to_bgr_uint8(raw)
    bgr320, crop = ts.preprocess_fundus(bgr, ts.CACHE_SIZE)
    rgb320 = cv2.cvtColor(bgr320, cv2.COLOR_BGR2RGB)
    res = ts.selective_predict(model, rgb320, meta['input_type'], meta['afep_params'], meta['temperature'],
                               meta['tau'], meta['n_small'], ts.AUG_SETS[meta['ttda_aug_set']], device)
    latency_ms = (time.perf_counter() - t0) * 1000
    x = ts.make_model_tensor(rgb320, meta['input_type'], meta['afep_params']).unsqueeze(0).to(device)
    cam, cls = ts.grad_cam(model, x)
    p_dr, unc = float(res['p_final'][1]), float(res['uncertainty'])

    c1, c2, c3 = st.columns(3)
    c1.image(rgb320, caption='Cropped fundus (320x320)')
    c2.image(ts.prepare_input(rgb320, meta['input_type'], meta['afep_params']), caption=f"Model input ({meta['input_type']})")
    c3.image(ts.cam_overlay(ts.prepare_input(rgb320, 'rgb'), cam), caption=f"Grad-CAM for class {'DR' if cls else 'No DR'}")
    a, b, c = st.columns(3)
    a.metric('Prediction', 'DR' if p_dr >= 0.5 else 'No DR')
    b.metric('Calibrated P(DR)', f'{p_dr:.3f}')
    c.metric('Uncertainty (entropy, bits)', f'{unc:.3f}')
    st.progress(min(max(unc, 0.0), 1.0))
    if unc > meta['referral_threshold']:
        st.warning('REFER to an ophthalmologist: uncertainty is above the referral threshold.')
    else:
        st.success('Automated decision: uncertainty is below the referral threshold.')
    st.caption(f"TTDA used: {res['used_ttda']} | forward passes: {res['n_forward']} | "
               f"end-to-end latency: {latency_ms:.0f} ms on {device_name} | crop ok: {crop['crop_ok']}")

In [ ]:
# 12.1 Consistency check
import importlib.util
sys.dont_write_bytecode = True
if (DEPLOY_DIR / 'trusteye_shared.py').exists() and SELECTION:
    spec_m = importlib.util.spec_from_file_location('trusteye_shared', DEPLOY_DIR / 'trusteye_shared.py')
    ts_mod = importlib.util.module_from_spec(spec_m); spec_m.loader.exec_module(ts_mod)
    meta = load_json(DEPLOY_DIR / 'model_meta.json')
    m_app = ts_mod.build_model(meta['activation'], meta['head_hidden'], meta['dropout'], pretrained=False)
    m_app.load_state_dict(torch.load(DEPLOY_DIR / 'model_selected.pt', map_location='cpu')); m_app.eval()
    m_nb, spec = load_model(SEL_DIR, torch.device('cpu'))
    diffs = []
    for im in x2[:5]:
        p_app = ts_mod.model_probs(m_app, ts_mod.make_model_tensor(im, meta['input_type'], meta['afep_params']).unsqueeze(0), meta['temperature'], torch.device('cpu'))
        p_nb = model_probs(m_nb, make_model_tensor(im, spec['input_type'], SEL_AFEP).unsqueeze(0), T_SEL, torch.device('cpu'))
        diffs.append(float(np.abs(p_app - p_nb).max()))
    print('max |p_app - p_notebook| over 5 val2 images:', max(diffs))
    assert max(diffs) < 1e-5, 'App code does not reproduce the notebook!'
    print('Consistency check passed.')

### 12.2 Launch the demo

In [ ]:
# 12.2 Launch the demo
LAUNCH_DEMO = False
if LAUNCH_DEMO and IN_COLAB:
    import re
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'])
    app_dir = Path('/content/trusteye_app')
    shutil.copytree(DEPLOY_DIR, app_dir, dirs_exist_ok=True)
    cf = Path('/content/cloudflared')
    if not cf.exists():
        run_cmd(['wget', '-q', '-O', cf, 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
        cf.chmod(0o755)
    subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                     cwd=app_dir, stdout=open('/content/streamlit.log', 'w'), stderr=subprocess.STDOUT)
    subprocess.Popen([str(cf), 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                     stdout=open('/content/cloudflared.log', 'w'), stderr=subprocess.STDOUT)
    url = None
    for _ in range(60):
        time.sleep(1)
        found = re.findall(r'https://[-a-z0-9]+\.trycloudflare\.com', Path('/content/cloudflared.log').read_text())
        if found:
            url = found[0]; break
    print('Open the demo:', url or 'not ready - check /content/cloudflared.log and /content/streamlit.log')
elif LAUNCH_DEMO:
    print('Not in Colab: run  streamlit run app.py  inside the deploy folder.')
else:
    print('Demo not launched (LAUNCH_DEMO = False).')

## Section 13 — P10: paper tables (filled only from saved results)

In [ ]:
# Section 13 — P10: paper tables (filled only from saved results)
def md_table(df):
    if df is None or len(df) == 0:
        return '_TBD (run pending)_\n'
    df = df.copy()
    for c in df.columns:
        if df[c].dtype.kind == 'f':
            df[c] = df[c].round(4)
    return '| ' + ' | '.join(map(str, df.columns)) + ' |\n|' + '---|' * len(df.columns) + '\n' + \
           '\n'.join('| ' + ' | '.join(map(str, r)) + ' |' for r in df.astype(str).values) + '\n'

def read_csv_or_none(p):
    return pd.read_csv(p) if Path(p).exists() else None

def read_json_or_none(p):
    return load_json(p) if Path(p).exists() else None

parts = [f'# TrustEye-Edge results (generated {datetime.now():%Y-%m-%d %H:%M}, RUN_MODE={RUN_MODE})\n',
         'All values below were produced by TrustEye_Edge.ipynb and read from files under results/ and runs/.\n']
p0 = read_json_or_none(VERIFY_JSON); dup = read_json_or_none(DUP_DIR / 'duplicate_stats.json'); spl = read_json_or_none(RESULTS_DIR / 'p0_split_report.json')
parts += ['## P0 data\n', f"- images verified: {p0['images']['n_decoded_ok'] if p0 else 'TBD (run pending)'}\n",
          f"- duplicate statistics: {json.dumps(dup) if dup else 'TBD (run pending)'}\n",
          f"- split counts: {spl['counts'] if spl else 'TBD (run pending)'}\n"]
parts += ['## P3 ablation (val1, mean over seeds)\n', md_table(read_csv_or_none(RESULTS_DIR / 'p3_ablation_runs.csv')
          .groupby(['input_type', 'activation'])[['accuracy', 'auc', 'ece', 'nll']].mean().reset_index()
          if (RESULTS_DIR / 'p3_ablation_runs.csv').exists() else None)]
parts += ['## P3 SiLU vs Mish tests (val1)\n', md_table(read_csv_or_none(RESULTS_DIR / 'p3_silu_vs_mish_tests.csv'))]
parts += ['## P4 temperature scaling\n', md_table(read_csv_or_none(RESULTS_DIR / 'p4_calibration.csv'))]
parts += ['## P5 TTDA (val2)\n', md_table(read_csv_or_none(RESULTS_DIR / 'p5_ttda_val2.csv'))]
parts += ['## P6 selective TTDA grid (val2)\n', md_table(read_csv_or_none(RESULTS_DIR / 'p6_selective_grid_val2.csv'))]
parts += ['## P6 AURC (val2)\n', md_table(read_csv_or_none(RESULTS_DIR / 'p6_aurc_val2.csv'))]
lat = read_json_or_none(LATENCY_FILE)
parts += ['## P6 latency (ms per image)\n', f"```\n{json.dumps(lat['results'], indent=1) if lat else 'TBD (run pending)'}\n```\n"]
parts += ['## P7 Grad-CAM sanity check\n', f"{read_json_or_none(RESULTS_DIR / 'p7_gradcam_sanity.json') or 'TBD (run pending)'}\n"]
for tag in ('test', 'idrid'):
    parts += [f'## P8 final metrics: {tag}\n', md_table(read_csv_or_none(FINAL_DIR / f'metrics_{tag}.csv')),
              f'### referral ({tag})\n', md_table(read_csv_or_none(FINAL_DIR / f'referral_{tag}.csv'))]
    fj = read_json_or_none(FINAL_DIR / f'metrics_{tag}.json')
    parts += [f"```\n{json.dumps(fj['statistics'], indent=1) if fj else 'TBD (run pending)'}\n```\n"]
parts += ['## Reported by Verma et al. (2026), Table 3 (IDRiD) - context only, not reproduced\n', md_table(PAPER_REPORTED)]
parts += ['## Deviations from the paper\n',
          '- V1 retina crop + pad to square before resizing\n- V2 own APTOS test split (+ IDRiD external test)\n'
          '- V3 OpenCV equivalents of the Keras augmentation\n- V4 explicit choices for unspecified AFEP details (Section 5)\n'
          '- V5 fine-tuning scope by EfficientNet stages, not "10 layers"\n- V6 entropy in bits; three uncertainty scores\n'
          '- V7 batch size, learning rates, early stopping chosen by us\n',
          '## Limitations\n',
          '- APTOS has no patient ids: different photos of the same eye cannot be grouped; rotated/flipped copies are not detected.\n'
          '- Binary task (No DR vs DR) hides severity grades.\n- Grad-CAM is qualitative.\n'
          '- CPU latency is a proxy, not an edge-device measurement.\n- Single dataset for training; IDRiD is small.\n']
(RESULTS_DIR / 'paper_tables.md').write_text(''.join(parts), encoding='utf-8')
from IPython.display import Markdown
display(Markdown(''.join(parts)))

### End of notebook